In [ ]:
from utils.visualization import *
from geometry.importers import STEPImporter
from solvers.frequency_domain import FrequencyDomainSolver
from rom.reduction import ModelOrderReduction
from analytical.rectangular_waveguide import RWGAnalytical
from analytical.cst_result import CSTResult
from ngsolve.webgui import Draw # must import Draw, otherwise may run into problems showing mesh
from utils.visualization import (
    plot_z_comparison, 
    plot_s_comparison, 
    plot_all_parameters,
    plot_eigenfrequencies,
    ParameterPlotter,
    EigenfrequencyPlotter
)
from core.persistence import *
from core.em_project import EMProject
from analytical.cst_result import CSTResult
%matplotlib widget
plt.rcParams['figure.dpi'] = 100

# project = EMProject("PillboxProject", 
# base_dir=r"C:\Users\Soske\Documents\git_projects\cavsim3d_simulations")

# # Import and setup geometry
# geo = project.import_geometry("pillbox/pillbox.step", auto_build=False)
# geo.add_splitting_plane_at_x(0.0)
# geo.split()
# geo.name_solids(sort_axis='X')
# geo.generate_mesh(maxh=0.04)

# # Solve
# result = project.fds.solve(fmin=0.1, fmax=2.5, nportmodes=1, order=1, rerun=True)
# roms = project.fds.foms.reduce()
# #import cst result
# cstresult = CSTResult(r'C:\Users\Soske\Documents\CST\pillbox')

# fig, ax = project.fds.foms.plot_s(['1(1)2(1)'])
# fig, ax = cstresult.plot_s(['1(1)2(1)'], ax=ax)


# roms_concat = roms.concatenate()
# print()
# result = roms_concat.solve(0.01, 2.0, 1000)
# print()
# roms_concat_rom = roms_concat.reduce()
# print()
# result = roms_concat_rom.solve(0.01, 2.0, 1000)
# print()
# import matplotlib.pyplot as plt
# # cstresult = CSTResult(r'C:\Users\Soske\Documents\CST\pillbox')

# # from analytical.circular_waveguide import CWGAnalytical
# # radius = 150e-3  # Width: 100 mm
# # L = 300e-3  # Length: 200 mm
# # cstresult = CWGAnalytical(radius=radius, length=L)
# which = ['1(1)1(1)']
# fig, axs = plt.subplot_mosaic([[1], [3]], layout='constrained', figsize=(6,8), sharex=True)

# cstresult.plot_s(['1(1)1(1)'], ax=axs[1], linewidth=2, label='CST Studio')
# roms_concat.plot_s(['1(1)1(1)'], ax=axs[1], label='Concatenated Reduced System')
# # roms_concat_rom.plot_s(['1(1)1(1)'], ax=axs[1])
# axs[1].set_ylabel(r'$|$ S1(1)1(1) $|$ [dB]')
# axs[1].set_title(r'')
# axs[1].set_xlabel(r'')

# cstresult.plot_s(['1(1)1(1)'], ax=axs[3], linewidth=2, plot_type='phase', label='CST Studio')
# roms_concat.plot_s(['1(1)1(1)'], ax=axs[3], plot_type='phase', label='Concatenated Reduced System')
# # roms_concat_rom.plot_s(['1(1)1(1)'], ax=axs[3], plot_type='phase')
# axs[3].set_ylabel(r'$\angle$ S1(1)1(1) [$^\circ$]')
# axs[3].set_title(r'')

# # plt.savefig('pillbox_sparameter.png', dpi=300)
# plt.show()

In [ ]:
%%timeit

project = EMProject("PillboxProject", 
base_dir=r"C:\Users\Soske\Documents\git_projects\cavsim3d_simulations")

# Import and setup geometry
geo = project.import_geometry("pillbox/pillbox.step", auto_build=False)
geo.add_splitting_plane_at_x(0.0)
geo.split()
geo.name_solids(sort_axis='X')
geo.generate_mesh(maxh=0.04)
result = project.fds.solve(fmin=0.1, fmax=2.5, nsamples=2, nportmodes=1, order=3, rerun=True)

In [ ]:
def func(x, y):
    return x + y

print(func)

In [ ]:
"""
Standalone Comprehensive Benchmark: CG vs GMRES with Various Preconditioners
for Electromagnetic (H(curl)) Problems in NGSolve

This script creates test problems and benchmarks different solver/preconditioner
combinations for frequency-domain electromagnetic simulations.

Author: Model Playground Assistant
"""

import numpy as np
import time
import os
from typing import Dict, List, Tuple, Optional
from dataclasses import dataclass, field
from collections import defaultdict
import matplotlib.pyplot as plt
import warnings

# NGSolve imports
from ngsolve import *
from netgen.occ import OCCGeometry, X, Y, Z
from ngsolve.webgui import Draw
from ngsolve import preconditioners

# Iterative: preconditioner MUST be registered before assembly
from ngsolve import Preconditioner as NGPreconditioner
from ngsolve import InnerProduct as NGInnerProduct
from ngsolve.krylovspace import GMRes, CG

# Output directory
OUTPUT_DIR = r"C:\Users\Soske\Documents\git_projects\cavsim3d_simulations\compare_cg_gmres"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
"""
Fixed Electromagnetic Solver Benchmark
With proper imports and error handling
"""

import numpy as np
import time
import os
from typing import Dict, List, Tuple, Optional
from dataclasses import dataclass, field
from collections import defaultdict
import matplotlib.pyplot as plt
import gc
import warnings

from ngsolve import *
from ngsolve import preconditioners

# CRITICAL: Import the Krylov solvers explicitly
from ngsolve.krylovspace import GMRes, CG

# Output directory
OUTPUT_DIR = r"C:\Users\Soske\Documents\git_projects\cavsim3d_simulations\compare_cg_gmres"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Physical constants
MU0 = 4 * np.pi * 1e-7
EPS0 = 8.854187817e-12


# =============================================================================
# Data Classes
# =============================================================================

@dataclass
class BenchmarkConfig:
    order: int = 1
    frequencies_ghz: np.ndarray = field(default_factory=lambda: np.linspace(1, 10, 10))
    tol: float = 1e-8
    maxiter: int = 2000
    restart: int = 100
    printrates: bool = False


@dataclass
class SolverResult:
    solver: str
    preconditioner: str
    strategy: str
    warm_start: bool
    frequency: float
    setup_time: float
    solve_time: float
    iterations: int
    residual: float
    converged: bool
    dofs: int
    failed: bool = False


# =============================================================================
# Mesh Generation
# =============================================================================

def create_waveguide_mesh(length=0.1, width=0.023, height=0.010, maxh=0.005):
    """Create rectangular waveguide mesh."""
    from netgen.occ import Box, OCCGeometry
    
    box = Box((0, 0, 0), (length, width, height))
    box.faces.Min(X).name = "port1"
    box.faces.Max(X).name = "port2"
    
    geo = OCCGeometry(box)
    ngmesh = geo.GenerateMesh(maxh=maxh)
    return Mesh(ngmesh)


def create_simple_box_mesh(size=0.1, maxh=0.02):
    """Create simple box mesh for testing."""
    from netgen.occ import Box, OCCGeometry
    
    box = Box((0, 0, 0), (size, size, size))
    geo = OCCGeometry(box)
    ngmesh = geo.GenerateMesh(maxh=maxh)
    return Mesh(ngmesh)


# =============================================================================
# Electromagnetic Problem
# =============================================================================

class ElectromagneticProblem:
    """Time-harmonic Maxwell problem."""
    
    def __init__(self, mesh: Mesh, order: int = 1):
        self.mesh = mesh
        self.order = order
        
        self.fes = HCurl(mesh, order=order, complex=False)
        self.u, self.v = self.fes.TnT()
        self.freedofs = self.fes.FreeDofs()
        self.n_free = sum(1 for i in range(self.fes.ndof) if self.freedofs[i])
        
        print(f"H(curl) space: order={order}, total DOFs={self.fes.ndof}, free DOFs={self.n_free}")
    
    def create_system_matrix(self, omega: float, sign: str = 'helmholtz') -> BilinearForm:
        """Create system matrix."""
        a = BilinearForm(self.fes)
        a += (1.0 / MU0) * curl(self.u) * curl(self.v) * dx
        
        if sign == 'helmholtz':
            a += -omega**2 * EPS0 * self.u * self.v * dx
        else:
            a += +omega**2 * EPS0 * self.u * self.v * dx
        
        return a
    
    def create_rhs(self) -> BaseVector:
        """Create RHS vector."""
        f = LinearForm(self.fes)
        f += InnerProduct(CF((1, 0, 0)), self.v) * dx
        
        with TaskManager():
            f.Assemble()
        
        vec = f.vec.CreateVector()
        vec.data = f.vec
        return vec


# =============================================================================
# Preconditioner Factory
# =============================================================================

def create_preconditioner(prec_type: str, bf: BilinearForm, **kwargs):
    """Create preconditioner."""
    prec_type = prec_type.lower()
    
    if prec_type in ['jacobi', 'local']:
        precond = preconditioners.Local(bf)
        desc = "Jacobi"
        
    elif prec_type in ['multigrid', 'mg']:
        precond = preconditioners.MultiGrid(
            bf,
            smoother=kwargs.get('smoother', 'point'),
            smoothingsteps=kwargs.get('smoothingsteps', 2),
            cycle=kwargs.get('cycle', 'V'),
            coarsetype=kwargs.get('coarsetype', 'direct'),
        )
        desc = "MG"
        
    elif prec_type == 'bddc':
        precond = preconditioners.BDDC(
            bf,
            coarsetype=kwargs.get('coarsetype', 'h1amg'),
        )
        desc = "BDDC"
        
    elif prec_type == 'none':
        precond = None
        desc = "None"
        
    else:
        raise ValueError(f"Unknown preconditioner: {prec_type}")
    
    return precond, desc


# =============================================================================
# Safe Solvers
# =============================================================================

def is_vector_valid(vec) -> bool:
    """Check if a vector is valid (not corrupted)."""
    try:
        arr = vec.FV().NumPy()
        if np.any(np.isnan(arr)) or np.any(np.isinf(arr)):
            return False
        return True
    except:
        return False


def safe_vector_copy(src, dst) -> bool:
    """Safely copy vector data. Returns True if successful."""
    try:
        if not is_vector_valid(src):
            return False
        dst.data = src
        return True
    except:
        return False


def compute_residual_safe(A_mat, x, b) -> float:
    """Safely compute residual. Returns 1.0 if failed."""
    try:
        if not is_vector_valid(x):
            return 1.0
        
        r = b.CreateVector()
        r.data = A_mat * x - b
        
        b_norm = Norm(b)
        r_norm = Norm(r)
        
        if b_norm > 0:
            return r_norm / b_norm
        else:
            return r_norm
    except:
        return 1.0


def solve_gmres_safe(A, b, precond=None, x0=None, tol=1e-8, maxiter=2000, restart=100):
    """
    Safe GMRES solver with proper error handling.
    
    Returns: (solution, iterations, residual, solve_time, success)
    """
    # Create solution vector
    x = b.CreateVector()
    x[:] = 0
    
    # Apply initial guess only if valid
    if x0 is not None and is_vector_valid(x0):
        try:
            x.data = x0
        except:
            x[:] = 0
    
    # Iteration counter
    iter_count = [0]
    def callback(r):
        iter_count[0] += 1
    
    pre_mat = precond.mat if precond is not None else None
    
    t_start = time.perf_counter()
    success = False
    result = None
    
    try:
        with TaskManager():
            result = GMRes(
                A=A.mat, 
                b=b, 
                x=x, 
                pre=pre_mat,
                maxsteps=maxiter, 
                tol=tol, 
                restart=restart,
                printrates=False, 
                callback=callback,
            )
        
        solve_time = time.perf_counter() - t_start
        
        # Check if result is valid
        if result is not None and is_vector_valid(result):
            x = result
            success = True
        else:
            x[:] = 0
            success = False
            
    except Exception as e:
        error_type = type(e).__name__
        error_msg = str(e)[:50] if str(e) else ""
        print(f"      GMRES exception: {error_type} {error_msg}")
        solve_time = time.perf_counter() - t_start
        x[:] = 0
        success = False
    
    # Compute residual safely
    if success:
        residual = compute_residual_safe(A.mat, x, b)
        if residual > 0.5:
            success = False
    else:
        residual = 1.0
        iter_count[0] = maxiter
    
    return x, iter_count[0], residual, solve_time, success


def solve_cg_safe(A, b, precond=None, x0=None, tol=1e-8, maxiter=2000):
    """
    Safe CG solver with proper error handling.
    
    Returns: (solution, iterations, residual, solve_time, success)
    """
    x = b.CreateVector()
    x[:] = 0
    
    if x0 is not None and is_vector_valid(x0):
        try:
            x.data = x0
        except:
            x[:] = 0
    
    iter_count = [0]
    def callback(r):
        iter_count[0] += 1
    
    pre_mat = precond.mat if precond is not None else None
    
    t_start = time.perf_counter()
    success = False
    result = None
    
    try:
        with TaskManager():
            result = CG(
                mat=A.mat, 
                rhs=b, 
                sol=x, 
                pre=pre_mat,
                maxsteps=maxiter, 
                tol=tol,
                printrates=False, 
                callback=callback,
                initialize=False,
            )
        
        solve_time = time.perf_counter() - t_start
        
        if result is not None and is_vector_valid(result):
            x = result
            success = True
        else:
            x[:] = 0
            success = False
            
    except Exception as e:
        error_type = type(e).__name__
        error_msg = str(e)[:50] if str(e) else ""
        print(f"      CG exception: {error_type} {error_msg}")
        solve_time = time.perf_counter() - t_start
        x[:] = 0
        success = False
    
    if success:
        residual = compute_residual_safe(A.mat, x, b)
        if residual > 0.5:
            success = False
    else:
        residual = 1.0
        iter_count[0] = maxiter
    
    return x, iter_count[0], residual, solve_time, success


# =============================================================================
# Benchmark Runner
# =============================================================================

def run_single_configuration(
    problem: ElectromagneticProblem,
    config: BenchmarkConfig,
    solver_type: str,
    prec_type: str,
    strategy: str,
    use_warm_start: bool,
    prec_kwargs: Dict = None,
) -> List[SolverResult]:
    """Run frequency sweep for a single configuration."""
    
    prec_kwargs = prec_kwargs or {}
    frequencies = config.frequencies_ghz * 1e9
    n_freq = len(frequencies)
    
    rhs_base = problem.create_rhs()
    results = []
    
    precond = None
    prec_desc = "None"
    total_setup_time = 0.0
    
    # For 'reuse_precond' strategy: build preconditioner once
    if strategy == 'reuse_precond' and prec_type != 'none':
        omega_ref = 2 * np.pi * frequencies[n_freq // 2]
        
        A_spd = problem.create_system_matrix(omega_ref, sign='spd')
        
        t_setup = time.perf_counter()
        precond, prec_desc = create_preconditioner(prec_type, A_spd, **prec_kwargs)
        
        with TaskManager():
            A_spd.Assemble()
        
        total_setup_time = time.perf_counter() - t_setup
        print(f"    Preconditioner setup (one-time): {total_setup_time:.3f}s")
    
    x_prev = None
    x_prev_valid = False
    consecutive_failures = 0
    max_consecutive_failures = 3
    
    for kk, freq in enumerate(frequencies):
        omega = 2 * np.pi * freq
        freq_ghz = freq / 1e9
        
        # Create system matrix
        A = problem.create_system_matrix(omega, sign='helmholtz')
        
        setup_time = 0.0
        
        # For 'rebuild' strategy: create new preconditioner
        if strategy == 'rebuild' and prec_type != 'none':
            t_setup = time.perf_counter()
            precond, prec_desc = create_preconditioner(prec_type, A, **prec_kwargs)
            setup_time = time.perf_counter() - t_setup
        
        # Assemble system matrix
        with TaskManager():
            A.Assemble()
        
        # Scale RHS
        rhs = rhs_base.CreateVector()
        rhs.data = omega * rhs_base
        
        # Determine initial guess
        x0 = None
        actually_used_warm = False
        
        if use_warm_start and x_prev is not None and x_prev_valid:
            if consecutive_failures < max_consecutive_failures:
                x0 = x_prev
                actually_used_warm = True
        
        # Solve
        if solver_type.lower() == 'gmres':
            x, iters, residual, solve_time, success = solve_gmres_safe(
                A, rhs, precond, x0=x0,
                tol=config.tol, maxiter=config.maxiter, restart=config.restart
            )
        else:
            x, iters, residual, solve_time, success = solve_cg_safe(
                A, rhs, precond, x0=x0,
                tol=config.tol, maxiter=config.maxiter
            )
        
        converged = success and (residual < config.tol * 10)
        
        # Update failure tracking
        if success:
            consecutive_failures = 0
        else:
            consecutive_failures += 1
        
        # Store solution for warm start (only if successful)
        if success and is_vector_valid(x):
            if x_prev is None:
                x_prev = rhs.CreateVector()
            x_prev_valid = safe_vector_copy(x, x_prev)
        else:
            x_prev_valid = False
        
        # Record result
        result = SolverResult(
            solver=solver_type.upper(),
            preconditioner=prec_desc,
            strategy=strategy,
            warm_start=actually_used_warm,
            frequency=freq_ghz,
            setup_time=setup_time if strategy == 'rebuild' else total_setup_time / n_freq,
            solve_time=solve_time,
            iterations=iters,
            residual=residual,
            converged=converged,
            dofs=problem.fes.ndof,
            failed=not success,
        )
        results.append(result)
        
        # Progress
        if (kk + 1) % max(1, n_freq // 5) == 0:
            if success:
                mark = "✓" if converged else "~"
            else:
                mark = "✗"
            warm_str = "W" if actually_used_warm else "C"
            print(f"      [{kk+1}/{n_freq}] {freq_ghz:.1f} GHz [{warm_str}]: {iters:5d} iter, "
                  f"res={residual:.2e}, t={solve_time:.3f}s {mark}")
        
        gc.collect()
    
    return results


def run_comprehensive_benchmark(problem: ElectromagneticProblem, 
                                 config: BenchmarkConfig) -> Dict[str, List[SolverResult]]:
    """Run all benchmark configurations."""
    
    all_results = {}
    
    configurations = [
        # GMRES with REBUILD strategy - Cold start
        ('gmres', 'jacobi', 'rebuild', False, {}, 'GMRES+Jacobi (rebuild, cold)'),
        ('gmres', 'multigrid', 'rebuild', False, {'smoothingsteps': 2}, 'GMRES+MG (rebuild, cold)'),
        ('gmres', 'bddc', 'rebuild', False, {}, 'GMRES+BDDC (rebuild, cold)'),
        
        # GMRES with REBUILD strategy - Warm start
        ('gmres', 'jacobi', 'rebuild', True, {}, 'GMRES+Jacobi (rebuild, warm)'),
        ('gmres', 'multigrid', 'rebuild', True, {'smoothingsteps': 2}, 'GMRES+MG (rebuild, warm)'),
        ('gmres', 'bddc', 'rebuild', True, {}, 'GMRES+BDDC (rebuild, warm)'),
        
        # GMRES with REUSE_PRECOND strategy
        ('gmres', 'jacobi', 'reuse_precond', True, {}, 'GMRES+Jacobi (reuse_P, warm)'),
        ('gmres', 'multigrid', 'reuse_precond', True, {'smoothingsteps': 2}, 'GMRES+MG (reuse_P, warm)'),
        ('gmres', 'bddc', 'reuse_precond', True, {}, 'GMRES+BDDC (reuse_P, warm)'),
        
        # CG (for comparison)
        ('cg', 'jacobi', 'rebuild', True, {}, 'CG+Jacobi (rebuild, warm)'),
        ('cg', 'multigrid', 'rebuild', True, {'smoothingsteps': 2}, 'CG+MG (rebuild, warm)'),
    ]
    
    for solver, prec, strategy, warm_start, prec_kwargs, name in configurations:
        print(f"\n{'='*65}")
        print(f"  {name}")
        print(f"  Solver: {solver.upper()}, Precond: {prec}, Strategy: {strategy}, Warm: {warm_start}")
        print(f"{'='*65}")
        
        try:
            results = run_single_configuration(
                problem, config,
                solver_type=solver,
                prec_type=prec,
                strategy=strategy,
                use_warm_start=warm_start,
                prec_kwargs=prec_kwargs,
            )
            all_results[name] = results
            
            # Summary
            total_time = sum(r.setup_time + r.solve_time for r in results)
            avg_iters = np.mean([r.iterations for r in results])
            n_conv = sum(1 for r in results if r.converged)
            n_fail = sum(1 for r in results if r.failed)
            
            print(f"\n    SUMMARY: {total_time:.2f}s total, {avg_iters:.1f} avg iter, "
                  f"{n_conv}/{len(results)} converged, {n_fail} failures")
            
        except Exception as e:
            print(f"    CONFIGURATION FAILED: {e}")
            import traceback
            traceback.print_exc()
        
        gc.collect()
    
    return all_results


# =============================================================================
# Visualization
# =============================================================================

def plot_results(all_results: Dict[str, List[SolverResult]], save_dir: str):
    """Create visualizations."""
    
    if not all_results:
        print("No results to plot!")
        return
    
    # Compute metrics
    metrics = {}
    for name, results in all_results.items():
        metrics[name] = {
            'total_time': sum(r.setup_time + r.solve_time for r in results),
            'avg_iters': np.mean([r.iterations for r in results]),
            'setup_time': sum(r.setup_time for r in results),
            'solve_time': sum(r.solve_time for r in results),
            'converged': sum(1 for r in results if r.converged),
            'failed': sum(1 for r in results if r.failed),
            'total': len(results),
            'freqs': [r.frequency for r in results],
            'iters': [r.iterations for r in results],
            'residuals': [r.residual for r in results],
        }
    
    # Sort by total time (failed configs at end)
    def sort_key(name):
        m = metrics[name]
        if m['failed'] > m['total'] // 2:
            return 1e10 + m['total_time']
        return m['total_time']
    
    sorted_names = sorted(metrics.keys(), key=sort_key)
    
    # =========================================================================
    # Figure 1: Summary comparison
    # =========================================================================
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # Colors based on failure rate
    colors = []
    for n in sorted_names:
        if metrics[n]['failed'] > metrics[n]['total'] // 2:
            colors.append('red')
        elif metrics[n]['failed'] > 0:
            colors.append('orange')
        else:
            colors.append('steelblue')
    
    # 1. Total time
    times = [metrics[n]['total_time'] for n in sorted_names]
    bars = axes[0, 0].barh(sorted_names, times, color=colors)
    axes[0, 0].set_xscale('log')
    axes[0, 0].set_xlabel('Total Time (seconds) - LOG SCALE')
    axes[0, 0].set_title('Total Execution Time\n(red=mostly failed, orange=some failures)')
    axes[0, 0].grid(True, alpha=0.3, axis='x', which='both')
    
    for bar, t in zip(bars, times):
        axes[0, 0].text(t * 1.1, bar.get_y() + bar.get_height()/2,
                        f'{t:.2f}s', va='center', fontsize=8)
    
    # 2. Average iterations
    avg_iters = [metrics[n]['avg_iters'] for n in sorted_names]
    bars = axes[0, 1].barh(sorted_names, avg_iters, color=colors)
    axes[0, 1].set_xscale('log')
    axes[0, 1].set_xlabel('Average Iterations - LOG SCALE')
    axes[0, 1].set_title('Average Iteration Count')
    axes[0, 1].grid(True, alpha=0.3, axis='x', which='both')
    
    # 3. Iterations vs Frequency
    for name in sorted_names:
        m = metrics[name]
        style = 'o-' if m['failed'] == 0 else 'x--'
        axes[1, 0].semilogy(m['freqs'], m['iters'], style, label=name, markersize=4, alpha=0.8)
    
    axes[1, 0].set_xlabel('Frequency (GHz)')
    axes[1, 0].set_ylabel('Iterations - LOG SCALE')
    axes[1, 0].set_title('Iterations vs Frequency')
    axes[1, 0].legend(fontsize=6, loc='upper left', bbox_to_anchor=(1.02, 1))
    axes[1, 0].grid(True, alpha=0.3, which='both')
    
    # 4. Time breakdown
    y_pos = np.arange(len(sorted_names))
    setup = [max(metrics[n]['setup_time'], 0.001) for n in sorted_names]
    solve = [max(metrics[n]['solve_time'], 0.001) for n in sorted_names]
    
    axes[1, 1].barh(y_pos, setup, label='Setup', color='steelblue')
    axes[1, 1].barh(y_pos, solve, left=setup, label='Solve', color='coral')
    axes[1, 1].set_yticks(y_pos)
    axes[1, 1].set_yticklabels(sorted_names, fontsize=8)
    axes[1, 1].set_xscale('log')
    axes[1, 1].set_xlabel('Time (seconds) - LOG SCALE')
    axes[1, 1].set_title('Time Breakdown: Setup vs Solve')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3, axis='x', which='both')
    
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, 'benchmark_summary_log.png'), dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Saved: benchmark_summary_log.png")
    
    # =========================================================================
    # Figure 2: Strategy comparison
    # =========================================================================
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    prec_types = ['Jacobi', 'MG', 'BDDC']
    
    for idx, prec in enumerate(prec_types):
        rebuild_names = [n for n in sorted_names if prec in n and 'rebuild' in n and 'warm' in n]
        reuse_names = [n for n in sorted_names if prec in n and 'reuse_P' in n]
        
        if rebuild_names:
            m = metrics[rebuild_names[0]]
            axes[idx].semilogy(m['freqs'], m['iters'], 'o-', label='Rebuild', 
                              color='steelblue', markersize=5)
        if reuse_names:
            m = metrics[reuse_names[0]]
            axes[idx].semilogy(m['freqs'], m['iters'], 's--', label='Reuse Precond', 
                              color='coral', markersize=5)
        
        axes[idx].set_xlabel('Frequency (GHz)')
        axes[idx].set_ylabel('Iterations (log)')
        axes[idx].set_title(f'{prec}: Rebuild vs Reuse Preconditioner')
        axes[idx].legend()
        axes[idx].grid(True, alpha=0.3, which='both')
    
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, 'strategy_comparison.png'), dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Saved: strategy_comparison.png")
    
    # =========================================================================
    # Figure 3: Cold vs Warm start
    # =========================================================================
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    for idx, prec in enumerate(prec_types):
        cold_names = [n for n in sorted_names if prec in n and 'cold' in n]
        warm_names = [n for n in sorted_names if prec in n and 'warm' in n and 'rebuild' in n]
        
        if cold_names:
            m = metrics[cold_names[0]]
            axes[idx].semilogy(m['freqs'], m['iters'], 'o-', label='Cold start', 
                              color='steelblue', markersize=5)
        if warm_names:
            m = metrics[warm_names[0]]
            axes[idx].semilogy(m['freqs'], m['iters'], 's--', label='Warm start', 
                              color='coral', markersize=5)
        
        axes[idx].set_xlabel('Frequency (GHz)')
        axes[idx].set_ylabel('Iterations (log)')
        axes[idx].set_title(f'{prec}: Cold vs Warm Start')
        axes[idx].legend()
        axes[idx].grid(True, alpha=0.3, which='both')
    
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, 'warmstart_comparison.png'), dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Saved: warmstart_comparison.png")


def save_results_text(all_results: Dict[str, List[SolverResult]], 
                      config: BenchmarkConfig, save_dir: str):
    """Save results to text file."""
    
    filepath = os.path.join(save_dir, 'benchmark_results.txt')
    
    with open(filepath, 'w') as f:
        f.write("=" * 100 + "\n")
        f.write("ELECTROMAGNETIC SOLVER BENCHMARK RESULTS\n")
        f.write("=" * 100 + "\n\n")
        
        f.write(f"Frequencies: {len(config.frequencies_ghz)} points, "
               f"{config.frequencies_ghz[0]:.1f} - {config.frequencies_ghz[-1]:.1f} GHz\n")
        f.write(f"Tolerance: {config.tol}\n")
        f.write(f"Max iterations: {config.maxiter}\n\n")
        
        sorted_items = sorted(all_results.items(), 
                             key=lambda x: sum(r.setup_time + r.solve_time for r in x[1]))
        
        f.write("-" * 100 + "\n")
        f.write(f"{'Configuration':<45} {'Total(s)':<10} {'AvgIter':<10} {'Conv':<8} {'Fail':<8}\n")
        f.write("-" * 100 + "\n")
        
        for name, results in sorted_items:
            total = sum(r.setup_time + r.solve_time for r in results)
            avg_it = np.mean([r.iterations for r in results])
            conv = sum(1 for r in results if r.converged)
            fail = sum(1 for r in results if r.failed)
            
            f.write(f"{name:<45} {total:<10.3f} {avg_it:<10.1f} {conv}/{len(results):<5} {fail}\n")
        
        f.write("\n" + "=" * 100 + "\n")
    
    print(f"Saved: {filepath}")


# =============================================================================
# Main
# =============================================================================

def main():
    print("=" * 70)
    print("ELECTROMAGNETIC SOLVER BENCHMARK")
    print("=" * 70)
    
    # Verify imports
    print("\nVerifying imports...")
    print(f"  GMRes: {GMRes}")
    print(f"  CG: {CG}")
    
    config = BenchmarkConfig(
        order=1,
        frequencies_ghz=np.linspace(1, 10, 10),
        tol=1e-8,
        maxiter=2000,
        restart=100,
    )
    
    # Create mesh
    print("\n1. Creating mesh...")
    try:
        mesh = create_waveguide_mesh(length=0.1, width=0.023, height=0.010, maxh=0.005)
        print(f"   Waveguide: {mesh.nv} vertices, {mesh.ne} elements")
    except Exception as e:
        print(f"   Waveguide mesh failed: {e}")
        mesh = create_simple_box_mesh(size=0.1, maxh=0.02)
        print(f"   Box: {mesh.nv} vertices, {mesh.ne} elements")
    
    # Setup problem
    print("\n2. Setting up problem...")
    problem = ElectromagneticProblem(mesh, order=config.order)
    
    # Run benchmark
    print("\n3. Running benchmarks...")
    all_results = run_comprehensive_benchmark(problem, config)
    
    # Generate outputs
    print("\n4. Generating visualizations...")
    plot_results(all_results, OUTPUT_DIR)
    save_results_text(all_results, config, OUTPUT_DIR)
    
    # Final summary
    print("\n" + "=" * 70)
    print("FINAL RANKING")
    print("=" * 70)
    
    def sort_key(item):
        name, results = item
        n_fail = sum(1 for r in results if r.failed)
        total_time = sum(r.setup_time + r.solve_time for r in results)
        if n_fail > len(results) // 2:
            return 1e10 + total_time
        return total_time
    
    sorted_results = sorted(all_results.items(), key=sort_key)
    
    print(f"\n{'Rank':<5} {'Configuration':<45} {'Time':<10} {'AvgIter':<10} {'Conv':<6} {'Fail':<6}")
    print("-" * 85)
    
    for rank, (name, results) in enumerate(sorted_results, 1):
        total = sum(r.setup_time + r.solve_time for r in results)
        avg_it = np.mean([r.iterations for r in results])
        n_conv = sum(1 for r in results if r.converged)
        n_fail = sum(1 for r in results if r.failed)
        status = "***" if n_fail > len(results) // 2 else ""
        print(f"{rank:<5} {name:<45} {total:<10.2f}s {avg_it:<10.1f} {n_conv:<6} {n_fail:<6} {status}")
    
    print("\n" + "=" * 70)
    print(f"Results saved to: {OUTPUT_DIR}")
    print("=" * 70)
    
    return all_results


if __name__ == "__main__":
    results = main()

In [ ]:
import matplotlib.pyplot as plt
# cstresult = CSTResult(r'C:\Users\Soske\Documents\CST\pillbox')

# from analytical.circular_waveguide import CWGAnalytical
# radius = 150e-3  # Width: 100 mm
# L = 300e-3  # Length: 200 mm
# cstresult = CWGAnalytical(radius=radius, length=L)
which = ['1(1)1(1)']
fig, axs = plt.subplot_mosaic([[1], [3]], layout='constrained', figsize=(6,8), sharex=True)

cstresult.plot_s(['1(1)1(1)'], ax=axs[1], linewidth=2, label='CST Studio')
roms_concat.plot_s(['1(1)1(1)'], ax=axs[1], label='Concatenated Reduced System')
# roms_concat_rom.plot_s(['1(1)1(1)'], ax=axs[1])
axs[1].set_ylabel(r'$|$ S1(1)1(1) $|$ [dB]')
axs[1].set_title(r'')
axs[1].set_xlabel(r'')

cstresult.plot_s(['1(1)1(1)'], ax=axs[3], linewidth=2, plot_type='phase', label='CST Studio')
roms_concat.plot_s(['1(1)1(1)'], ax=axs[3], plot_type='phase', label='Concatenated Reduced System')
# roms_concat_rom.plot_s(['1(1)1(1)'], ax=axs[3], plot_type='phase')
axs[3].set_ylabel(r'$\angle$ S1(1)1(1) [$^\circ$]')
axs[3].set_title(r'')

# plt.savefig('pillbox_sparameter.png', dpi=300)
plt.show()

In [ ]:
# project.fds.foms.plot_s()

In [ ]:
# import matplotlib.pyplot as plt
# from analytical.cst_result import CSTResult
# cstresult = CSTResult(r'C:\Users\Soske\Documents\CST\pillbox')

# # from analytical.circular_waveguide import CWGAnalytical
# # radius = 150e-3  # Width: 100 mm
# # L = 300e-3  # Length: 200 mm
# # cstresult = CWGAnalytical(radius=radius, length=L)
# which = ['1(1)1(1)']
# fig, axs = plt.subplot_mosaic([[1], [3]], layout='constrained', figsize=(6,8), sharex=True)

# cstresult.plot_s(['1(1)1(1)'], ax=axs[1], linewidth=2, label='CST Studio')
# project.fds.foms.roms.concat.plot_s(['1(1)1(1)'], ax=axs[1], label='Concatenated Reduced System')
# # roms_concat_rom.plot_s(['1(1)1(1)'], ax=axs[1])
# axs[1].set_ylabel(r'$|$ S1(1)1(1) $|$ [dB]')
# axs[1].set_title(r'')
# axs[1].set_xlabel(r'')

# cstresult.plot_s(['1(1)1(1)'], ax=axs[3], linewidth=2, plot_type='phase', label='CST Studio')
# project.fds.foms.roms.concat.plot_s(['1(1)1(1)'], ax=axs[3], plot_type='phase', label='Concatenated Reduced System')
# # roms_concat_rom.plot_s(['1(1)1(1)'], ax=axs[3], plot_type='phase')
# axs[3].set_ylabel(r'$\angle$ S1(1)1(1) [$^\circ$]')
# axs[3].set_title(r'')

# # plt.savefig('pillbox_sparameter.png', dpi=300)
# plt.show()

In [ ]:
# project.geo.mesh.GetMaterials()
# project.fds.plot_field(90, domain='cell_1')

In [ ]:
# project.geo.show('mesh')

In [ ]:
from ngsolve.webgui import Draw # must import Draw, otherwise may run into problems showing mesh
from utils.visualization import (
    plot_z_comparison, 
    plot_s_comparison, 
    plot_all_parameters,
    plot_eigenfrequencies,
    ParameterPlotter,
    EigenfrequencyPlotter
)
from core.persistence import *
from core.em_project import EMProject
from analytical.cst_result import CSTResult
import matplotlib.pyplot as plt
%matplotlib widget
plt.rcParams['figure.dpi'] = 100

project = EMProject("PillboxProject", 
base_dir=r"C:\Users\Soske\Documents\git_projects\cavsim3d_simulations")

In [ ]:
"""
CG vs GMRES Solver Comparison for SPD Matrices
================================================
Compares Conjugate Gradient and GMRES iterative solvers across:
  - Matrix sizes:      small (100), medium (500), large (2000)
  - Condition numbers: well-conditioned (κ~10), moderate (κ~1e4), ill-conditioned (κ~1e8)
  - Preconditioners:   none, Jacobi (diagonal), ILU(0) (direct sparse), 
                       and a 2-level Multigrid (geometric/algebraic-style)

Dependencies: numpy, scipy, matplotlib
"""

import time
import warnings
import numpy as np
import scipy.sparse as sp
import scipy.sparse.linalg as spla
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import LogNorm
from dataclasses import dataclass, field
from typing import Optional

warnings.filterwarnings("ignore", category=sp.SparseEfficiencyWarning)

# ─────────────────────────────────────────────────────────────────────────────
# 1.  Matrix generation
# ─────────────────────────────────────────────────────────────────────────────

def make_spd_matrix(n: int, condition_number: float, seed: int = 42) -> sp.csr_matrix:
    """
    Build a sparse SPD matrix of size n×n with the requested condition number.

    Strategy
    --------
    Start from a 1-D Poisson stencil (which is already SPD and sparse) and
    *shift + scale* its eigenvalues so that λ_min = 1 and λ_max ≈ condition_number.
    This keeps the sparsity pattern realistic while giving exact control over κ.
    """
    rng = np.random.default_rng(seed)

    # 1-D Poisson: tridiagonal [-1, 2, -1], size n
    A_base = sp.diags(
        [-np.ones(n - 1), 2 * np.ones(n), -np.ones(n - 1)],
        [-1, 0, 1], shape=(n, n), format="csr"
    )

    # Eigenvalues of 1-D Poisson: λ_k = 2 - 2cos(kπ/(n+1)), k=1..n
    # λ_min ≈ (π/(n+1))², λ_max ≈ 4
    lam_min_base = 2 - 2 * np.cos(np.pi / (n + 1))
    lam_max_base = 2 - 2 * np.cos(n * np.pi / (n + 1))

    # Shift so that λ_min → 1
    shift = 1.0 - lam_min_base
    A_shifted = A_base + shift * sp.eye(n, format="csr")
    lam_min_s = 1.0
    lam_max_s = lam_max_base + shift

    # Scale so that λ_max → condition_number (λ_min stays 1)
    # We need a diagonal perturbation that maps the spectrum:
    # new λ_k = α * (λ_k - lam_min_s) + lam_min_s   with α = (kappa-1)/(lam_max_s-1)
    alpha = (condition_number - 1.0) / max(lam_max_s - lam_min_s, 1e-10)
    # A_scaled = alpha*(A_shifted - I) + I  = alpha*A_shifted + (1-alpha)*I
    A_scaled = alpha * A_shifted + (1.0 - alpha) * sp.eye(n, format="csr")

    # Add a small random diagonal bump to break perfect symmetry of Poisson,
    # but keep the matrix SPD (bumps are positive).
    bump = rng.uniform(0.0, 0.05, n)
    A_final = A_scaled + sp.diags(bump, 0, shape=(n, n), format="csr")

    return A_final.tocsr()


# ─────────────────────────────────────────────────────────────────────────────
# 2.  Preconditioner factory
# ─────────────────────────────────────────────────────────────────────────────

def jacobi_preconditioner(A: sp.csr_matrix) -> spla.LinearOperator:
    """M = diag(A)^{-1}  (point Jacobi / diagonal scaling)."""
    d = A.diagonal()
    d_inv = np.where(np.abs(d) > 1e-14, 1.0 / d, 1.0)
    n = A.shape[0]
    return spla.LinearOperator((n, n), matvec=lambda x: d_inv * x, dtype=float)


def ilu_preconditioner(A: sp.csr_matrix) -> spla.LinearOperator:
    """ILU(0) factorisation via scipy's spilu — a direct sparse preconditioner."""
    ilu = spla.spilu(A.tocsc(), fill_factor=1, drop_tol=1e-4)
    n = A.shape[0]
    return spla.LinearOperator((n, n), matvec=ilu.solve, dtype=float)


def multigrid_preconditioner(A: sp.csr_matrix, levels: int = 3) -> spla.LinearOperator:
    """
    Algebraic two-grid (V-cycle) preconditioner.

    Coarsening: keep every other degree of freedom (injection-style aggregation).
    Smoother:   weighted Jacobi (ω = 2/3).
    Coarse solve: direct (spsolve).

    For matrices larger than ~2000 we do a recursive coarsening; for smaller
    matrices a single level suffices.
    """
    n = A.shape[0]

    def weighted_jacobi(A_, r, omega=2.0 / 3.0, iterations=2):
        d = A_.diagonal()
        d_inv = np.where(np.abs(d) > 1e-14, 1.0 / d, 1.0)
        x = np.zeros_like(r)
        for _ in range(iterations):
            x = x + omega * d_inv * (r - A_.dot(x))
        return x

    def build_prolongation(n_fine):
        """Linear interpolation P: coarse → fine (size n_fine × n_coarse)."""
        n_coarse = (n_fine + 1) // 2
        rows, cols, vals = [], [], []
        for i in range(n_fine):
            c = i // 2
            if i % 2 == 0:                      # coarse node
                rows.append(i); cols.append(c); vals.append(1.0)
            else:                               # fine node: average neighbours
                rows.append(i); cols.append(c); vals.append(0.5)
                if c + 1 < n_coarse:
                    rows.append(i); cols.append(c + 1); vals.append(0.5)
        P = sp.csr_matrix((vals, (rows, cols)), shape=(n_fine, n_coarse))
        return P

    def vcycle(A_f, r, depth):
        if A_f.shape[0] <= 50 or depth == 0:
            return spla.spsolve(A_f, r)

        # Pre-smooth
        x = weighted_jacobi(A_f, r)
        residual = r - A_f.dot(x)

        # Restrict
        P = build_prolongation(A_f.shape[0])
        R = P.T
        A_c = (R @ A_f @ P).tocsr()
        r_c = R.dot(residual)

        # Coarse correction (recursive)
        e_c = vcycle(A_c, r_c, depth - 1)

        # Prolongate
        x = x + P.dot(e_c)

        # Post-smooth
        x = weighted_jacobi(A_f, r - A_f.dot(x) + r) * 0 + x   # re-smooth
        x = x + weighted_jacobi(A_f, r - A_f.dot(x))

        return x

    effective_levels = min(levels, max(1, int(np.log2(n)) - 3))
    return spla.LinearOperator(
        (n, n),
        matvec=lambda r: vcycle(A, r, effective_levels),
        dtype=float,
    )


PRECONDITIONERS = {
    "None":      lambda A: None,
    "Jacobi":    jacobi_preconditioner,
    "ILU(0)":    ilu_preconditioner,
    "Multigrid": multigrid_preconditioner,
}


# ─────────────────────────────────────────────────────────────────────────────
# 3.  Solver wrappers with iteration counting
# ─────────────────────────────────────────────────────────────────────────────

@dataclass
class SolverResult:
    solver:     str
    precond:    str
    n:          int
    kappa:      float
    kappa_label: str
    iterations: int
    time_sec:   float
    residual:   float
    converged:  bool


def _make_counter():
    count = [0]
    def callback(_):
        count[0] += 1
    return callback, count


def run_cg(A, b, M, rtol=1e-8, maxiter=5000):
    cb, cnt = _make_counter()
    t0 = time.perf_counter()
    x, info = spla.cg(A, b, M=M, rtol=rtol, maxiter=maxiter, callback=cb)
    elapsed = time.perf_counter() - t0
    res = np.linalg.norm(b - A.dot(x)) / (np.linalg.norm(b) + 1e-30)
    return elapsed, cnt[0], res, (info == 0)


def run_gmres(A, b, M, rtol=1e-8, maxiter=5000, restart=50):
    count = [0]
    def cb(r): count[0] += 1
    t0 = time.perf_counter()
    x, info = spla.gmres(A, b, M=M, rtol=rtol, maxiter=maxiter,
                         restart=restart, callback=cb, callback_type='pr_norm')
    elapsed = time.perf_counter() - t0
    res = np.linalg.norm(b - A.dot(x)) / (np.linalg.norm(b) + 1e-30)
    return elapsed, count[0], res, (info == 0)


# ─────────────────────────────────────────────────────────────────────────────
# 4.  Experiment grid
# ─────────────────────────────────────────────────────────────────────────────

SIZES  = [100, 500, 2000]
KAPPAS = {
    "Well (κ≈10)":       1e1,
    "Moderate (κ≈1e4)":  1e4,
    "Ill (κ≈1e8)":       1e8,
}
SOLVERS = {"CG": run_cg, "GMRES": run_gmres}


def run_all_experiments() -> list[SolverResult]:
    results = []
    total = len(SIZES) * len(KAPPAS) * len(PRECONDITIONERS) * len(SOLVERS)
    done  = 0

    for n in SIZES:
        for kappa_label, kappa in KAPPAS.items():
            A = make_spd_matrix(n, kappa)
            rng = np.random.default_rng(0)
            b = rng.standard_normal(n)
            b /= np.linalg.norm(b)

            for prec_name, prec_fn in PRECONDITIONERS.items():
                # Build preconditioner (may fail for ill-conditioned ILU)
                try:
                    M = prec_fn(A)
                except Exception:
                    M = None

                for solver_name, solver_fn in SOLVERS.items():
                    done += 1
                    print(f"  [{done:3d}/{total}] n={n:5d}  κ={kappa_label:20s}"
                          f"  prec={prec_name:10s}  solver={solver_name} ... ", end="", flush=True)
                    try:
                        t, iters, res, ok = solver_fn(A, b, M)
                        flag = "OK" if ok else f"NOT CONV (res={res:.1e})"
                        print(f"{flag}  iters={iters}  t={t:.3f}s")
                    except Exception as e:
                        t, iters, res, ok = 0.0, -1, np.nan, False
                        print(f"ERROR: {e}")

                    results.append(SolverResult(
                        solver=solver_name, precond=prec_name,
                        n=n, kappa=kappa, kappa_label=kappa_label,
                        iterations=iters, time_sec=t,
                        residual=res, converged=ok,
                    ))
    return results


# ─────────────────────────────────────────────────────────────────────────────
# 5.  Plotting
# ─────────────────────────────────────────────────────────────────────────────

SOLVER_COLORS = {"CG": "#2563eb", "GMRES": "#dc2626"}
PREC_MARKERS  = {"None": "o", "Jacobi": "s", "ILU(0)": "^", "Multigrid": "D"}
PREC_LINESTYLE = {"None": "--", "Jacobi": "-.", "ILU(0)": "-", "Multigrid": ":"}


def _filter(results, **kwargs):
    out = results
    for k, v in kwargs.items():
        out = [r for r in out if getattr(r, k) == v]
    return out


def plot_vs_size(results, metric="iterations"):
    """
    For each (kappa, preconditioner) pair, show CG vs GMRES as a function of n.
    """
    fig, axes = plt.subplots(
        len(KAPPAS), len(PRECONDITIONERS),
        figsize=(18, 11), sharey="row",
    )
    fig.suptitle(
        f"CG vs GMRES — {metric.replace('_',' ').title()} vs Matrix Size",
        fontsize=15, fontweight="bold", y=1.01,
    )

    ylabel = "Iterations" if metric == "iterations" else "Time (s)"

    for row, (kappa_label, kappa) in enumerate(KAPPAS.items()):
        for col, prec_name in enumerate(PRECONDITIONERS):
            ax = axes[row][col]
            for solver_name in SOLVERS:
                xs, ys = [], []
                for n in SIZES:
                    r_list = _filter(results, solver=solver_name, precond=prec_name,
                                     n=n, kappa_label=kappa_label)
                    if r_list:
                        r = r_list[0]
                        v = getattr(r, metric)
                        xs.append(n); ys.append(v if r.converged else np.nan)
                ax.plot(xs, ys,
                        color=SOLVER_COLORS[solver_name],
                        marker=PREC_MARKERS[prec_name],
                        linestyle=PREC_LINESTYLE[prec_name],
                        linewidth=2, markersize=8,
                        label=solver_name)
                # Mark non-converged with ×
                for x_, y_, r_ in zip(xs, ys, [_filter(results, solver=solver_name,
                                                         precond=prec_name, n=x__,
                                                         kappa_label=kappa_label)[0]
                                                for x__ in xs]):
                    if not r_.converged:
                        ax.plot(x_, y_ if not np.isnan(y_) else 0,
                                'x', color='k', markersize=14, markeredgewidth=2)

            ax.set_title(f"{prec_name}\n{kappa_label}", fontsize=9)
            ax.set_xlabel("Matrix size n")
            if col == 0:
                ax.set_ylabel(ylabel)
            ax.set_xticks(SIZES)
            ax.grid(True, alpha=0.3)
            ax.legend(fontsize=8)

    fig.tight_layout()
    return fig


def plot_heatmap(results, metric="iterations", solver_name="CG"):
    """
    Heatmap: rows = condition numbers, cols = sizes, one subplot per preconditioner.
    """
    kappa_labels = list(KAPPAS.keys())
    prec_names   = list(PRECONDITIONERS.keys())

    fig, axes = plt.subplots(1, len(prec_names), figsize=(16, 4))
    fig.suptitle(
        f"{solver_name} — {metric.replace('_',' ').title()} Heatmap",
        fontsize=14, fontweight="bold",
    )

    for ax, prec_name in zip(axes, prec_names):
        mat = np.full((len(kappa_labels), len(SIZES)), np.nan)
        for i, kl in enumerate(kappa_labels):
            for j, n in enumerate(SIZES):
                r_list = _filter(results, solver=solver_name, precond=prec_name,
                                 n=n, kappa_label=kl)
                if r_list and r_list[0].converged:
                    mat[i, j] = getattr(r_list[0], metric)

        vmin = np.nanmin(mat[mat > 0]) if not np.all(np.isnan(mat)) else 1
        vmax = np.nanmax(mat) if not np.all(np.isnan(mat)) else 10
        if vmin == vmax:
            vmax = vmin + 1
        im = ax.imshow(mat, aspect="auto",
                       norm=LogNorm(vmin=max(vmin, 1e-6), vmax=vmax),
                       cmap="plasma")
        ax.set_xticks(range(len(SIZES)));       ax.set_xticklabels(SIZES)
        ax.set_yticks(range(len(kappa_labels))); ax.set_yticklabels(
            [kl.split()[0] for kl in kappa_labels], fontsize=8)
        ax.set_title(f"Precond: {prec_name}", fontsize=10)
        ax.set_xlabel("Matrix size n")
        if prec_name == prec_names[0]:
            ax.set_ylabel("Condition number")
        for i in range(mat.shape[0]):
            for j in range(mat.shape[1]):
                val = mat[i, j]
                txt = f"{val:.0f}" if not np.isnan(val) else "✗"
                ax.text(j, i, txt, ha="center", va="center",
                        fontsize=8, color="white", fontweight="bold")
        plt.colorbar(im, ax=ax, shrink=0.8,
                     label="Iterations" if metric == "iterations" else "Time (s)")

    fig.tight_layout()
    return fig


def plot_speedup_bar(results):
    """
    Bar chart: speedup of each preconditioner over 'No preconditioner',
    for both CG and GMRES, grouped by condition number (for n=2000).
    """
    n_ref   = max(SIZES)
    prec_names = [p for p in PRECONDITIONERS if p != "None"]
    kappa_labels = list(KAPPAS.keys())

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(
        f"Preconditioner Speed-up (iterations) vs 'No Preconditioner'  [n={n_ref}]",
        fontsize=13, fontweight="bold",
    )
    x = np.arange(len(kappa_labels))
    width = 0.25

    for ax, solver_name in zip(axes, SOLVERS):
        ax.set_title(solver_name, fontsize=12)
        for k, prec_name in enumerate(prec_names):
            speedups = []
            for kl in kappa_labels:
                base = _filter(results, solver=solver_name, precond="None",
                               n=n_ref, kappa_label=kl)
                prec = _filter(results, solver=solver_name, precond=prec_name,
                               n=n_ref, kappa_label=kl)
                if base and prec and base[0].converged and prec[0].converged:
                    s = base[0].iterations / max(prec[0].iterations, 1)
                else:
                    s = 0.0
                speedups.append(s)
            offset = (k - len(prec_names) / 2 + 0.5) * width
            bars = ax.bar(x + offset, speedups, width,
                          label=prec_name, zorder=3)
            for bar, sv in zip(bars, speedups):
                if sv > 0:
                    ax.text(bar.get_x() + bar.get_width() / 2,
                            bar.get_height() + 0.05, f"{sv:.1f}×",
                            ha="center", va="bottom", fontsize=8)

        ax.axhline(1.0, color="k", linestyle="--", linewidth=1, label="Baseline (×1)")
        ax.set_xticks(x)
        ax.set_xticklabels([kl.split()[0] + "\n" + kl.split("(")[1].rstrip(")")
                            for kl in kappa_labels], fontsize=9)
        ax.set_ylabel("Speed-up  (iter_none / iter_prec)")
        ax.set_ylim(bottom=0)
        ax.legend(fontsize=9)
        ax.grid(axis="y", alpha=0.3, zorder=0)

    fig.tight_layout()
    return fig


def plot_convergence_profile(results_conv: dict):
    """
    Plot residual norm vs iteration for selected (n, kappa, preconditioner) combos.
    results_conv is a dict keyed by (solver, precond, n, kappa_label) → list of residuals.
    """
    kappa_label = "Moderate (κ≈1e4)"
    n = 500
    prec_list = list(PRECONDITIONERS.keys())

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle(
        f"Convergence History  [n={n}, {kappa_label}]",
        fontsize=13, fontweight="bold",
    )

    for ax, solver_name in zip(axes, ["CG", "GMRES"]):
        ax.set_title(solver_name)
        for prec_name in prec_list:
            key = (solver_name, prec_name, n, kappa_label)
            if key in results_conv:
                hist = results_conv[key]
                ax.semilogy(hist,
                            label=prec_name,
                            marker=PREC_MARKERS[prec_name],
                            linestyle=PREC_LINESTYLE[prec_name],
                            markevery=max(1, len(hist) // 10))
        ax.set_xlabel("Iteration")
        ax.set_ylabel("Relative residual")
        ax.legend()
        ax.grid(True, which="both", alpha=0.3)

    fig.tight_layout()
    return fig


# ─────────────────────────────────────────────────────────────────────────────
# 6.  Convergence history collection
# ─────────────────────────────────────────────────────────────────────────────

def collect_convergence_history(n=500, kappa_label="Moderate (κ≈1e4)"):
    kappa = KAPPAS[kappa_label]
    A = make_spd_matrix(n, kappa)
    rng = np.random.default_rng(0)
    b = rng.standard_normal(n); b /= np.linalg.norm(b)
    conv_data = {}

    for prec_name, prec_fn in PRECONDITIONERS.items():
        try:
            M = prec_fn(A)
        except Exception:
            M = None

        for solver_name in ["CG", "GMRES"]:
            history = []
            def cb(xk):
                r = np.linalg.norm(b - A.dot(xk)) / (np.linalg.norm(b) + 1e-30)
                history.append(r)

            try:
                if solver_name == "CG":
                    spla.cg(A, b, M=M, rtol=1e-10, maxiter=3000, callback=cb)
                else:
                    hist_gmres = []
                    def gmres_cb(pr):
                        hist_gmres.append(float(pr) * (np.linalg.norm(b) + 1e-30) / (np.linalg.norm(b) + 1e-30))
                    spla.gmres(A, b, M=M, rtol=1e-10, maxiter=3000,
                               restart=50, callback=gmres_cb, callback_type='pr_norm')
                    history.extend(hist_gmres)
            except Exception:
                pass
            conv_data[(solver_name, prec_name, n, kappa_label)] = history

    return conv_data


# ─────────────────────────────────────────────────────────────────────────────
# 7.  Summary table
# ─────────────────────────────────────────────────────────────────────────────

def print_summary_table(results):
    header = (f"{'Solver':<8} {'Precond':<12} {'n':>6} {'κ':>10} "
              f"{'Iters':>7} {'Time(s)':>9} {'Residual':>12} {'Conv?':>6}")
    print("\n" + "=" * len(header))
    print("RESULTS SUMMARY")
    print("=" * len(header))
    print(header)
    print("-" * len(header))

    prev_kl = None
    for r in results:
        if r.kappa_label != prev_kl:
            print(f"\n  ── {r.kappa_label} ──")
            prev_kl = r.kappa_label
        iters_str = str(r.iterations) if r.converged else f"{r.iterations}*"
        print(f"  {r.solver:<8} {r.precond:<12} {r.n:>6} {r.kappa:>10.0e} "
              f"{iters_str:>7} {r.time_sec:>9.4f} {r.residual:>12.3e} "
              f"{'✓' if r.converged else '✗':>6}")
    print("=" * len(header))
    print("  * Not converged within maxiter\n")


# ─────────────────────────────────────────────────────────────────────────────
# 8.  Main
# ─────────────────────────────────────────────────────────────────────────────

def main():
    print("=" * 65)
    print("  CG vs GMRES — SPD Matrix Solver Comparison")
    print("=" * 65)
    print(f"  Matrix sizes : {SIZES}")
    print(f"  κ values     : {list(KAPPAS.values())}")
    print(f"  Solvers      : {list(SOLVERS.keys())}")
    print(f"  Preconditioners: {list(PRECONDITIONERS.keys())}")
    print("=" * 65 + "\n")

    # --- Run experiments ---
    print("Running solver experiments …\n")
    results = run_all_experiments()

    # --- Summary table ---
    print_summary_table(results)

    # --- Convergence history ---
    print("\nCollecting convergence history for n=500, κ≈1e4 …")
    conv_history = collect_convergence_history(n=500, kappa_label="Moderate (κ≈1e4)")

    # ── Figures ──────────────────────────────────────────────────────────────
    print("\nGenerating plots …")

    fig1 = plot_vs_size(results, metric="iterations")
    fig1.savefig(r"C:\Users\Soske\Documents\git_projects\cavsim3d_simulations\compare_cg_gmres/fig1_iterations_vs_size.png",
                 dpi=150, bbox_inches="tight")

    fig2 = plot_vs_size(results, metric="time_sec")
    fig2.savefig(r"C:\Users\Soske\Documents\git_projects\cavsim3d_simulations\compare_cg_gmres/fig2_time_vs_size.png",
                 dpi=150, bbox_inches="tight")

    fig3 = plot_heatmap(results, metric="iterations", solver_name="CG")
    fig3.savefig(r"C:\Users\Soske\Documents\git_projects\cavsim3d_simulations\compare_cg_gmres/fig3_heatmap_CG_iters.png",
                 dpi=150, bbox_inches="tight")

    fig4 = plot_heatmap(results, metric="iterations", solver_name="GMRES")
    fig4.savefig(r"C:\Users\Soske\Documents\git_projects\cavsim3d_simulations\compare_cg_gmres/fig4_heatmap_GMRES_iters.png",
                 dpi=150, bbox_inches="tight")

    fig5 = plot_speedup_bar(results)
    fig5.savefig(r"C:\Users\Soske\Documents\git_projects\cavsim3d_simulations\compare_cg_gmres/fig5_precond_speedup.png",
                 dpi=150, bbox_inches="tight")

    fig6 = plot_convergence_profile(conv_history)
    fig6.savefig(r"C:\Users\Soske\Documents\git_projects\cavsim3d_simulations\compare_cg_gmres/fig6_convergence_history.png",
                 dpi=150, bbox_inches="tight")

    print("\nAll figures saved to /mnt/user-data/outputs/")
    print("Done.")


if __name__ == "__main__":
    main()

In [ ]:
"""
Comprehensive Comparison of CG and GMRES Methods
for SPD Matrices with Various Condition Numbers and Preconditioners

Author: Model Playground Assistant
"""

import numpy as np
import scipy.sparse as sp
from scipy.sparse.linalg import cg, gmres, spilu, LinearOperator, splu
from scipy.sparse import diags, kron, eye
import matplotlib.pyplot as plt
import time
import os
from collections import defaultdict
import warnings
from typing import Dict, List, Tuple, Optional, Callable
from dataclasses import dataclass
import json

warnings.filterwarnings('ignore')

# Try to import pyamg for multigrid preconditioners
try:
    import pyamg
    PYAMG_AVAILABLE = True
except ImportError:
    PYAMG_AVAILABLE = False
    print("WARNING: pyamg not available. Install with 'pip install pyamg' for multigrid tests.")

# Output directory
OUTPUT_DIR = r"C:\Users\Soske\Documents\git_projects\cavsim3d_simulations\compare_cg_gmres"
os.makedirs(OUTPUT_DIR, exist_ok=True)


# =============================================================================
# Data Classes and Helper Functions
# =============================================================================

@dataclass
class SolverResult:
    """Store results from a solver run"""
    method: str
    matrix_type: str
    matrix_size: int
    condition_number: float
    preconditioner: str
    time: float
    iterations: int
    converged: bool
    residual_norm: float
    residual_history: List[float]


class IterationCounter:
    """Callback to count iterations and store residuals"""
    def __init__(self):
        self.iterations = 0
        self.residuals = []
    
    def __call__(self, rk):
        self.iterations += 1
        if isinstance(rk, np.ndarray):
            self.residuals.append(np.linalg.norm(rk))
        else:
            self.residuals.append(abs(rk))


# =============================================================================
# Matrix Generation Functions
# =============================================================================

def generate_spd_matrix_eigenvalues(n: int, condition_number: float, 
                                     sparse: bool = True) -> sp.csr_matrix:
    """
    Generate SPD matrix with specified condition number using eigenvalue control.
    For sparse matrices, we use a modified approach.
    """
    if sparse and n > 500:
        # For large sparse matrices, use a different approach
        return generate_modified_laplacian(n, condition_number)
    
    # Generate random orthogonal matrix
    np.random.seed(42)
    Q, _ = np.linalg.qr(np.random.randn(n, n))
    
    # Create eigenvalues with desired condition number
    eigenvalues = np.linspace(1, condition_number, n)
    
    # Form A = Q * D * Q^T
    A = Q @ np.diag(eigenvalues) @ Q.T
    
    # Ensure symmetry
    A = (A + A.T) / 2
    
    if sparse:
        return sp.csr_matrix(A)
    return A


def generate_1d_laplacian(n: int) -> sp.csr_matrix:
    """Generate 1D Laplacian matrix (tridiagonal)"""
    return diags([-1, 2, -1], [-1, 0, 1], shape=(n, n), format='csr')


def generate_2d_laplacian(n: int) -> sp.csr_matrix:
    """Generate 2D Laplacian matrix (5-point stencil)"""
    n_sqrt = int(np.sqrt(n))
    if n_sqrt * n_sqrt != n:
        n_sqrt = int(np.ceil(np.sqrt(n)))
    
    # 1D Laplacian
    T = diags([-1, 2, -1], [-1, 0, 1], shape=(n_sqrt, n_sqrt), format='csr')
    I = eye(n_sqrt, format='csr')
    
    # 2D Laplacian via Kronecker product
    A = kron(I, T) + kron(T, I)
    return A


def generate_3d_laplacian(n: int) -> sp.csr_matrix:
    """Generate 3D Laplacian matrix (7-point stencil)"""
    n_cbrt = int(np.round(n ** (1/3)))
    
    T = diags([-1, 2, -1], [-1, 0, 1], shape=(n_cbrt, n_cbrt), format='csr')
    I = eye(n_cbrt, format='csr')
    
    # 2D Laplacian
    A2D = kron(I, T) + kron(T, I)
    I2D = eye(n_cbrt * n_cbrt, format='csr')
    
    # 3D Laplacian
    A = kron(I, A2D) + kron(kron(T, I), I)
    return A[:n, :n]


def generate_modified_laplacian(n: int, condition_number: float) -> sp.csr_matrix:
    """
    Generate modified Laplacian with approximately specified condition number.
    Adds a diagonal shift to control conditioning.
    """
    # Start with 2D Laplacian
    A = generate_2d_laplacian(n)
    
    # Estimate spectral bounds
    lambda_max_approx = 8.0  # For 2D Laplacian
    
    # Add shift to achieve desired condition number
    # kappa = lambda_max / lambda_min
    # lambda_min = lambda_max / kappa
    lambda_min_target = lambda_max_approx / condition_number
    
    # Add diagonal shift
    shift = max(0, lambda_min_target)
    A = A + shift * eye(A.shape[0], format='csr')
    
    return A


def generate_anisotropic_laplacian(n: int, epsilon: float = 0.01) -> sp.csr_matrix:
    """
    Generate anisotropic diffusion matrix.
    Small epsilon leads to ill-conditioning.
    """
    n_sqrt = int(np.sqrt(n))
    if n_sqrt * n_sqrt != n:
        n_sqrt = int(np.ceil(np.sqrt(n)))
    
    # Anisotropic coefficients
    Tx = diags([-1, 2, -1], [-1, 0, 1], shape=(n_sqrt, n_sqrt), format='csr')
    Ty = diags([-epsilon, 2*epsilon, -epsilon], [-1, 0, 1], 
               shape=(n_sqrt, n_sqrt), format='csr')
    I = eye(n_sqrt, format='csr')
    
    A = kron(I, Tx) + kron(Ty, I)
    actual_size = A.shape[0]
    
    if actual_size > n:
        A = A[:n, :n]
    
    return A


def generate_random_spd(n: int, density: float = 0.1, 
                        condition_number: float = 10.0) -> sp.csr_matrix:
    """Generate random sparse SPD matrix"""
    np.random.seed(42)
    
    # Generate random sparse matrix
    B = sp.random(n, n, density=density, format='csr')
    
    # Make it SPD: A = B^T * B + shift * I
    A = B.T @ B
    
    # Add diagonal dominance for conditioning control
    diag_shift = condition_number / 10
    A = A + diag_shift * eye(n, format='csr')
    
    return A


def generate_elasticity_matrix(n: int) -> sp.csr_matrix:
    """
    Generate a simplified elasticity-type matrix.
    Block structure similar to linear elasticity problems.
    """
    n_per_dim = int(np.sqrt(n // 2))
    if n_per_dim < 2:
        n_per_dim = 2
    
    # Stiffness-like matrix
    K = generate_2d_laplacian(n_per_dim * n_per_dim)
    
    # Coupling term
    C = 0.3 * K
    
    # Block matrix
    actual_n = n_per_dim * n_per_dim
    A = sp.bmat([[K, C], [C, K]], format='csr')
    
    # Trim to desired size
    final_size = min(A.shape[0], n)
    return A[:final_size, :final_size]


def estimate_condition_number(A: sp.csr_matrix, num_eigenvalues: int = 10) -> float:
    """Estimate condition number using sparse eigenvalue computation"""
    try:
        from scipy.sparse.linalg import eigsh
        n = A.shape[0]
        k = min(num_eigenvalues, n - 2)
        
        if k < 2:
            return np.nan
        
        # Largest eigenvalues
        lambda_max = eigsh(A, k=1, which='LM', return_eigenvectors=False)[0]
        
        # Smallest eigenvalues
        lambda_min = eigsh(A, k=1, which='SM', return_eigenvectors=False)[0]
        
        if lambda_min > 0:
            return abs(lambda_max / lambda_min)
        else:
            return np.inf
    except:
        return np.nan


# =============================================================================
# Preconditioner Functions
# =============================================================================

def get_jacobi_preconditioner(A: sp.csr_matrix) -> LinearOperator:
    """Jacobi (diagonal) preconditioner"""
    diag = A.diagonal()
    diag[diag == 0] = 1.0  # Avoid division by zero
    
    def matvec(x):
        return x / diag
    
    return LinearOperator(A.shape, matvec=matvec)


def get_ssor_preconditioner(A: sp.csr_matrix, omega: float = 1.0) -> LinearOperator:
    """SSOR preconditioner"""
    D = diags(A.diagonal(), 0, format='csr')
    L = sp.tril(A, -1, format='csr')
    
    def matvec(x):
        # Forward sweep
        y = np.zeros_like(x)
        for i in range(len(x)):
            row_start = L.indptr[i]
            row_end = L.indptr[i+1]
            s = x[i]
            for j in range(row_start, row_end):
                s -= omega * L.data[j] * y[L.indices[j]]
            y[i] = s / A.diagonal()[i]
        
        # Backward sweep
        z = np.zeros_like(x)
        for i in range(len(x)-1, -1, -1):
            z[i] = y[i]
        
        return z
    
    return LinearOperator(A.shape, matvec=matvec)


def get_ilu_preconditioner(A: sp.csr_matrix, fill_factor: int = 10) -> Optional[LinearOperator]:
    """ILU preconditioner (Incomplete LU factorization)"""
    try:
        ilu = spilu(A.tocsc(), fill_factor=fill_factor)
        return LinearOperator(A.shape, matvec=ilu.solve)
    except:
        return None


def get_ichol_preconditioner(A: sp.csr_matrix) -> Optional[LinearOperator]:
    """
    Incomplete Cholesky preconditioner (approximation using ILU).
    For true IC, specialized libraries are needed.
    """
    try:
        # Use ILU as approximation (scipy doesn't have direct IC)
        ilu = spilu(A.tocsc(), fill_factor=5, drop_tol=1e-4)
        return LinearOperator(A.shape, matvec=ilu.solve)
    except:
        return None


def get_amg_preconditioner(A: sp.csr_matrix, 
                           strength: str = 'symmetric',
                           max_coarse: int = 50) -> Optional[LinearOperator]:
    """Algebraic Multigrid preconditioner using pyamg"""
    if not PYAMG_AVAILABLE:
        return None
    
    try:
        # Build AMG hierarchy
        ml = pyamg.smoothed_aggregation_solver(A, 
                                                strength=strength,
                                                max_coarse=max_coarse)
        return ml.aspreconditioner()
    except:
        return None


def get_ruge_stuben_amg_preconditioner(A: sp.csr_matrix) -> Optional[LinearOperator]:
    """Ruge-Stuben AMG preconditioner"""
    if not PYAMG_AVAILABLE:
        return None
    
    try:
        ml = pyamg.ruge_stuben_solver(A, max_coarse=50)
        return ml.aspreconditioner()
    except:
        return None


def get_direct_preconditioner(A: sp.csr_matrix) -> Optional[LinearOperator]:
    """Direct solver as preconditioner (for comparison)"""
    try:
        lu = splu(A.tocsc())
        return LinearOperator(A.shape, matvec=lu.solve)
    except:
        return None


# =============================================================================
# Solver Wrapper Functions
# =============================================================================

def solve_cg(A: sp.csr_matrix, b: np.ndarray, 
             M: Optional[LinearOperator] = None,
             rtol: float = 1e-10, 
             maxiter: int = 10000) -> SolverResult:
    """Wrapper for CG solver with timing and iteration counting"""
    
    counter = IterationCounter()
    
    start_time = time.perf_counter()
    x, info = cg(A, b, M=M, rtol=rtol, maxiter=maxiter, callback=counter)
    end_time = time.perf_counter()
    
    residual_norm = np.linalg.norm(b - A @ x) / np.linalg.norm(b)
    
    return SolverResult(
        method='CG',
        matrix_type='',
        matrix_size=A.shape[0],
        condition_number=0,
        preconditioner='',
        time=end_time - start_time,
        iterations=counter.iterations,
        converged=(info == 0),
        residual_norm=residual_norm,
        residual_history=counter.residuals
    )


def solve_gmres(A: sp.csr_matrix, b: np.ndarray,
                M: Optional[LinearOperator] = None,
                rtol: float = 1e-10,
                maxiter: int = 10000,
                restart: int = 50) -> SolverResult:
    """Wrapper for GMRES solver with timing and iteration counting"""
    
    counter = IterationCounter()
    
    start_time = time.perf_counter()
    x, info = gmres(A, b, M=M, rtol=rtol, maxiter=maxiter, 
                    restart=restart, callback=counter, callback_type='pr_norm')
    end_time = time.perf_counter()
    
    residual_norm = np.linalg.norm(b - A @ x) / np.linalg.norm(b)
    
    return SolverResult(
        method='GMRES',
        matrix_type='',
        matrix_size=A.shape[0],
        condition_number=0,
        preconditioner='',
        time=end_time - start_time,
        iterations=counter.iterations,
        converged=(info == 0),
        residual_norm=residual_norm,
        residual_history=counter.residuals
    )


# =============================================================================
# Comparison Functions
# =============================================================================

def run_single_comparison(A: sp.csr_matrix, 
                          matrix_type: str,
                          condition_number: float,
                          preconditioner_name: str,
                          M: Optional[LinearOperator],
                          rtol: float = 1e-10,
                          maxiter: int = 10000) -> Tuple[SolverResult, SolverResult]:
    """Run CG and GMRES on the same system"""
    
    n = A.shape[0]
    
    # Create RHS vector (using a known solution for verification)
    np.random.seed(123)
    x_true = np.random.randn(n)
    b = A @ x_true
    
    # Run CG
    cg_result = solve_cg(A, b, M=M, rtol=rtol, maxiter=maxiter)
    cg_result.matrix_type = matrix_type
    cg_result.condition_number = condition_number
    cg_result.preconditioner = preconditioner_name
    
    # Run GMRES
    gmres_result = solve_gmres(A, b, M=M, rtol=rtol, maxiter=maxiter)
    gmres_result.matrix_type = matrix_type
    gmres_result.condition_number = condition_number
    gmres_result.preconditioner = preconditioner_name
    
    return cg_result, gmres_result


def run_comprehensive_comparison():
    """Run comprehensive comparison across all configurations"""
    
    print("=" * 80)
    print("COMPREHENSIVE CG vs GMRES COMPARISON")
    print("=" * 80)
    
    # Configuration
    sizes = [1e1, 1e2, 1e3, 1e4, 1e5]
    condition_numbers = [10, 100, 1000, 10000, 100000, 1e6]  # Well to ill-conditioned
    
    matrix_generators = {
        '2D Laplacian': generate_2d_laplacian,
        '3D Laplacian': generate_3d_laplacian,
        'Anisotropic (eps=0.1)': lambda n: generate_anisotropic_laplacian(n, 0.1),
        'Anisotropic (eps=0.01)': lambda n: generate_anisotropic_laplacian(n, 0.01),
        'Random SPD': lambda n: generate_random_spd(n, density=0.05),
        'Elasticity': generate_elasticity_matrix,
    }
    
    # Store all results
    all_results: List[SolverResult] = []
    
    # ==========================================================================
    # Test 1: Different matrix sizes with standard 2D Laplacian
    # ==========================================================================
    print("\n" + "=" * 60)
    print("TEST 1: Matrix Size Comparison (2D Laplacian)")
    print("=" * 60)
    
    for size in sizes:
        print(f"\n  Size: {size}")
        A = generate_2d_laplacian(size)
        cond = estimate_condition_number(A)
        
        for prec_name, prec_func in [('None', lambda A: None),
                                      ('Jacobi', get_jacobi_preconditioner),
                                      ('ILU', get_ilu_preconditioner)]:
            M = prec_func(A)
            cg_res, gmres_res = run_single_comparison(A, '2D Laplacian', cond, prec_name, M)
            all_results.extend([cg_res, gmres_res])
            print(f"    {prec_name:10s} | CG: {cg_res.iterations:5d} iter, {cg_res.time:.4f}s | "
                  f"GMRES: {gmres_res.iterations:5d} iter, {gmres_res.time:.4f}s")
    
    # ==========================================================================
    # Test 2: Different condition numbers
    # ==========================================================================
    print("\n" + "=" * 60)
    print("TEST 2: Condition Number Comparison")
    print("=" * 60)
    
    size_for_cond_test = 1000
    
    for target_cond in condition_numbers:
        print(f"\n  Target κ ≈ {target_cond}")
        A = generate_modified_laplacian(size_for_cond_test, target_cond)
        actual_cond = estimate_condition_number(A)
        
        for prec_name, prec_func in [('None', lambda A: None),
                                      ('Jacobi', get_jacobi_preconditioner),
                                      ('ILU', get_ilu_preconditioner)]:
            M = prec_func(A)
            cg_res, gmres_res = run_single_comparison(
                A, f'Modified Laplacian (κ≈{target_cond})', 
                actual_cond if not np.isnan(actual_cond) else target_cond, 
                prec_name, M
            )
            all_results.extend([cg_res, gmres_res])
            print(f"    {prec_name:10s} | CG: {cg_res.iterations:5d} iter, {cg_res.time:.4f}s | "
                  f"GMRES: {gmres_res.iterations:5d} iter, {gmres_res.time:.4f}s")
    
    # ==========================================================================
    # Test 3: Different matrix types
    # ==========================================================================
    print("\n" + "=" * 60)
    print("TEST 3: Matrix Type Comparison")
    print("=" * 60)
    
    size_for_type_test = 1000
    
    for mat_name, mat_func in matrix_generators.items():
        print(f"\n  Matrix: {mat_name}")
        try:
            A = mat_func(size_for_type_test)
            cond = estimate_condition_number(A)
            
            for prec_name, prec_func in [('None', lambda A: None),
                                          ('ILU', get_ilu_preconditioner)]:
                M = prec_func(A)
                cg_res, gmres_res = run_single_comparison(A, mat_name, cond, prec_name, M)
                all_results.extend([cg_res, gmres_res])
                print(f"    {prec_name:10s} | CG: {cg_res.iterations:5d} iter, {cg_res.time:.4f}s | "
                      f"GMRES: {gmres_res.iterations:5d} iter, {gmres_res.time:.4f}s")
        except Exception as e:
            print(f"    Error: {e}")
    
    # ==========================================================================
    # Test 4: Preconditioner Comparison
    # ==========================================================================
    print("\n" + "=" * 60)
    print("TEST 4: Comprehensive Preconditioner Comparison")
    print("=" * 60)
    
    size_for_prec_test = 2000
    A = generate_2d_laplacian(size_for_prec_test)
    cond = estimate_condition_number(A)
    
    preconditioners = {
        'None': lambda A: None,
        'Jacobi': get_jacobi_preconditioner,
        'ILU (fill=5)': lambda A: get_ilu_preconditioner(A, fill_factor=5),
        'ILU (fill=20)': lambda A: get_ilu_preconditioner(A, fill_factor=20),
        'IC (approx)': get_ichol_preconditioner,
    }
    
    if PYAMG_AVAILABLE:
        preconditioners['AMG (SA)'] = get_amg_preconditioner
        preconditioners['AMG (RS)'] = get_ruge_stuben_amg_preconditioner
    
    print(f"\n  Matrix: 2D Laplacian ({size_for_prec_test}x{size_for_prec_test})")
    
    for prec_name, prec_func in preconditioners.items():
        try:
            M = prec_func(A)
            cg_res, gmres_res = run_single_comparison(A, '2D Laplacian', cond, prec_name, M)
            all_results.extend([cg_res, gmres_res])
            print(f"    {prec_name:15s} | CG: {cg_res.iterations:5d} iter, {cg_res.time:.4f}s | "
                  f"GMRES: {gmres_res.iterations:5d} iter, {gmres_res.time:.4f}s")
        except Exception as e:
            print(f"    {prec_name:15s} | Error: {e}")
    
    # ==========================================================================
    # Test 5: AMG vs Direct Preconditioner Scaling
    # ==========================================================================
    if PYAMG_AVAILABLE:
        print("\n" + "=" * 60)
        print("TEST 5: AMG Scaling Study")
        print("=" * 60)
        
        amg_sizes = [500, 1000, 2000, 4000, 8000]
        
        for size in amg_sizes:
            print(f"\n  Size: {size}")
            A = generate_2d_laplacian(size)
            cond = estimate_condition_number(A)
            
            # No preconditioner
            cg_none, gmres_none = run_single_comparison(A, '2D Laplacian', cond, 'None', None)
            print(f"    None:       CG: {cg_none.iterations:5d} iter, {cg_none.time:.4f}s | "
                  f"GMRES: {gmres_none.iterations:5d} iter, {gmres_none.time:.4f}s")
            all_results.extend([cg_none, gmres_none])
            
            # AMG preconditioner
            M_amg = get_amg_preconditioner(A)
            if M_amg is not None:
                cg_amg, gmres_amg = run_single_comparison(A, '2D Laplacian', cond, 'AMG (SA)', M_amg)
                print(f"    AMG (SA):   CG: {cg_amg.iterations:5d} iter, {cg_amg.time:.4f}s | "
                      f"GMRES: {gmres_amg.iterations:5d} iter, {gmres_amg.time:.4f}s")
                all_results.extend([cg_amg, gmres_amg])
    
    # ==========================================================================
    # Test 6: Ill-conditioned systems comparison
    # ==========================================================================
    print("\n" + "=" * 60)
    print("TEST 6: Ill-Conditioned Systems")
    print("=" * 60)
    
    for eps in [0.1, 0.01, 0.001, 0.0001]:
        print(f"\n  Anisotropy epsilon = {eps}")
        A = generate_anisotropic_laplacian(1000, eps)
        cond = estimate_condition_number(A)
        
        for prec_name, prec_func in [('None', lambda A: None),
                                      ('ILU', get_ilu_preconditioner)]:
            M = prec_func(A)
            cg_res, gmres_res = run_single_comparison(
                A, f'Anisotropic (eps={eps})', cond, prec_name, M, maxiter=20000
            )
            all_results.extend([cg_res, gmres_res])
            conv_cg = '✓' if cg_res.converged else '✗'
            conv_gmres = '✓' if gmres_res.converged else '✗'
            print(f"    {prec_name:10s} | CG: {cg_res.iterations:5d} iter {conv_cg}, {cg_res.time:.4f}s | "
                  f"GMRES: {gmres_res.iterations:5d} iter {conv_gmres}, {gmres_res.time:.4f}s")
        
        if PYAMG_AVAILABLE:
            M_amg = get_amg_preconditioner(A)
            if M_amg is not None:
                cg_res, gmres_res = run_single_comparison(
                    A, f'Anisotropic (eps={eps})', cond, 'AMG', M_amg, maxiter=20000
                )
                all_results.extend([cg_res, gmres_res])
                conv_cg = '✓' if cg_res.converged else '✗'
                conv_gmres = '✓' if gmres_res.converged else '✗'
                print(f"    {'AMG':10s} | CG: {cg_res.iterations:5d} iter {conv_cg}, {cg_res.time:.4f}s | "
                      f"GMRES: {gmres_res.iterations:5d} iter {conv_gmres}, {gmres_res.time:.4f}s")
    
    return all_results


# =============================================================================
# Visualization Functions
# =============================================================================

def create_size_comparison_plots(results: List[SolverResult]):
    """Create plots comparing performance across matrix sizes"""
    
    # Filter results for 2D Laplacian
    laplacian_results = [r for r in results if '2D Laplacian' in r.matrix_type]
    
    if not laplacian_results:
        return
    
    # Group by preconditioner
    preconditioners = list(set(r.preconditioner for r in laplacian_results))
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 12))
    
    for prec in preconditioners:
        prec_results = [r for r in laplacian_results if r.preconditioner == prec]
        
        cg_results = [r for r in prec_results if r.method == 'CG']
        gmres_results = [r for r in prec_results if r.method == 'GMRES']
        
        # Sort by size
        cg_results.sort(key=lambda x: x.matrix_size)
        gmres_results.sort(key=lambda x: x.matrix_size)
        
        if cg_results:
            sizes_cg = [r.matrix_size for r in cg_results]
            times_cg = [r.time for r in cg_results]
            iters_cg = [r.iterations for r in cg_results]
            
            axes[0, 0].plot(sizes_cg, times_cg, 'o-', label=f'CG ({prec})')
            axes[0, 1].plot(sizes_cg, iters_cg, 'o-', label=f'CG ({prec})')
        
        if gmres_results:
            sizes_gmres = [r.matrix_size for r in gmres_results]
            times_gmres = [r.time for r in gmres_results]
            iters_gmres = [r.iterations for r in gmres_results]
            
            axes[1, 0].plot(sizes_gmres, times_gmres, 's--', label=f'GMRES ({prec})')
            axes[1, 1].plot(sizes_gmres, iters_gmres, 's--', label=f'GMRES ({prec})')
    
    axes[0, 0].set_xlabel('Matrix Size')
    axes[0, 0].set_ylabel('Time (seconds)')
    axes[0, 0].set_title('CG: Execution Time vs Matrix Size')
    axes[0, 0].legend()
    axes[0, 0].set_xscale('log')
    axes[0, 0].set_yscale('log')
    axes[0, 0].grid(True, alpha=0.3)
    
    axes[0, 1].set_xlabel('Matrix Size')
    axes[0, 1].set_ylabel('Iterations')
    axes[0, 1].set_title('CG: Iterations vs Matrix Size')
    axes[0, 1].legend()
    axes[0, 1].set_xscale('log')
    axes[0, 1].grid(True, alpha=0.3)
    
    axes[1, 0].set_xlabel('Matrix Size')
    axes[1, 0].set_ylabel('Time (seconds)')
    axes[1, 0].set_title('GMRES: Execution Time vs Matrix Size')
    axes[1, 0].legend()
    axes[1, 0].set_xscale('log')
    axes[1, 0].set_yscale('log')
    axes[1, 0].grid(True, alpha=0.3)
    
    axes[1, 1].set_xlabel('Matrix Size')
    axes[1, 1].set_ylabel('Iterations')
    axes[1, 1].set_title('GMRES: Iterations vs Matrix Size')
    axes[1, 1].legend()
    axes[1, 1].set_xscale('log')
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'size_comparison.png'), dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Saved: size_comparison.png")


def create_condition_number_plots(results: List[SolverResult]):
    """Create plots comparing performance across condition numbers"""
    
    # Filter results for modified Laplacian
    cond_results = [r for r in results if 'Modified Laplacian' in r.matrix_type]
    
    if not cond_results:
        return
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 12))
    
    preconditioners = list(set(r.preconditioner for r in cond_results))
    colors = plt.cm.tab10(np.linspace(0, 1, len(preconditioners)))
    
    for idx, prec in enumerate(preconditioners):
        prec_results = [r for r in cond_results if r.preconditioner == prec]
        
        cg_results = [r for r in prec_results if r.method == 'CG']
        gmres_results = [r for r in prec_results if r.method == 'GMRES']
        
        # Sort by condition number
        cg_results.sort(key=lambda x: x.condition_number)
        gmres_results.sort(key=lambda x: x.condition_number)
        
        if cg_results:
            conds_cg = [r.condition_number for r in cg_results]
            times_cg = [r.time for r in cg_results]
            iters_cg = [r.iterations for r in cg_results]
            
            axes[0, 0].plot(conds_cg, times_cg, 'o-', color=colors[idx], label=f'CG ({prec})')
            axes[0, 1].plot(conds_cg, iters_cg, 'o-', color=colors[idx], label=f'CG ({prec})')
        
        if gmres_results:
            conds_gmres = [r.condition_number for r in gmres_results]
            times_gmres = [r.time for r in gmres_results]
            iters_gmres = [r.iterations for r in gmres_results]
            
            axes[1, 0].plot(conds_gmres, times_gmres, 's--', color=colors[idx], label=f'GMRES ({prec})')
            axes[1, 1].plot(conds_gmres, iters_gmres, 's--', color=colors[idx], label=f'GMRES ({prec})')
    
    for ax in axes.flat:
        ax.set_xscale('log')
        ax.grid(True, alpha=0.3)
    
    axes[0, 0].set_xlabel('Condition Number')
    axes[0, 0].set_ylabel('Time (seconds)')
    axes[0, 0].set_title('CG: Time vs Condition Number')
    axes[0, 0].legend()
    axes[0, 0].set_yscale('log')
    
    axes[0, 1].set_xlabel('Condition Number')
    axes[0, 1].set_ylabel('Iterations')
    axes[0, 1].set_title('CG: Iterations vs Condition Number')
    axes[0, 1].legend()
    
    axes[1, 0].set_xlabel('Condition Number')
    axes[1, 0].set_ylabel('Time (seconds)')
    axes[1, 0].set_title('GMRES: Time vs Condition Number')
    axes[1, 0].legend()
    axes[1, 0].set_yscale('log')
    
    axes[1, 1].set_xlabel('Condition Number')
    axes[1, 1].set_ylabel('Iterations')
    axes[1, 1].set_title('GMRES: Iterations vs Condition Number')
    axes[1, 1].legend()
    
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'condition_number_comparison.png'), dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Saved: condition_number_comparison.png")


def create_preconditioner_comparison_plots(results: List[SolverResult]):
    """Create bar charts comparing different preconditioners"""
    
    # Filter for 2D Laplacian results
    laplacian_results = [r for r in results if '2D Laplacian' in r.matrix_type]
    
    # Get unique sizes
    sizes = sorted(list(set(r.matrix_size for r in laplacian_results)))
    
    if len(sizes) < 1:
        return
    
    # Use largest size for comparison
    target_size = max(sizes)
    size_results = [r for r in laplacian_results if r.matrix_size == target_size]
    
    if not size_results:
        return
    
    preconditioners = list(set(r.preconditioner for r in size_results))
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    x = np.arange(len(preconditioners))
    width = 0.35
    
    cg_times = []
    gmres_times = []
    cg_iters = []
    gmres_iters = []
    
    for prec in preconditioners:
        prec_results = [r for r in size_results if r.preconditioner == prec]
        cg_res = [r for r in prec_results if r.method == 'CG']
        gmres_res = [r for r in prec_results if r.method == 'GMRES']
        
        cg_times.append(cg_res[0].time if cg_res else 0)
        gmres_times.append(gmres_res[0].time if gmres_res else 0)
        cg_iters.append(cg_res[0].iterations if cg_res else 0)
        gmres_iters.append(gmres_res[0].iterations if gmres_res else 0)
    
    axes[0].bar(x - width/2, cg_times, width, label='CG', color='steelblue')
    axes[0].bar(x + width/2, gmres_times, width, label='GMRES', color='coral')
    axes[0].set_xlabel('Preconditioner')
    axes[0].set_ylabel('Time (seconds)')
    axes[0].set_title(f'Execution Time Comparison (n={target_size})')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(preconditioners, rotation=45, ha='right')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3, axis='y')
    
    axes[1].bar(x - width/2, cg_iters, width, label='CG', color='steelblue')
    axes[1].bar(x + width/2, gmres_iters, width, label='GMRES', color='coral')
    axes[1].set_xlabel('Preconditioner')
    axes[1].set_ylabel('Iterations')
    axes[1].set_title(f'Iteration Count Comparison (n={target_size})')
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(preconditioners, rotation=45, ha='right')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'preconditioner_comparison.png'), dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Saved: preconditioner_comparison.png")


def create_matrix_type_comparison_plots(results: List[SolverResult]):
    """Create comparison plots for different matrix types"""
    
    matrix_types = list(set(r.matrix_type for r in results))
    matrix_types = [mt for mt in matrix_types if 'Modified' not in mt]  # Exclude modified Laplacians
    
    if len(matrix_types) < 2:
        return
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # Group by matrix type, without preconditioner
    no_prec_results = [r for r in results if r.preconditioner == 'None']
    
    cg_times = {}
    gmres_times = {}
    cg_iters = {}
    gmres_iters = {}
    
    for mt in matrix_types:
        mt_results = [r for r in no_prec_results if r.matrix_type == mt]
        cg_res = [r for r in mt_results if r.method == 'CG']
        gmres_res = [r for r in mt_results if r.method == 'GMRES']
        
        if cg_res:
            cg_times[mt] = cg_res[0].time
            cg_iters[mt] = cg_res[0].iterations
        if gmres_res:
            gmres_times[mt] = gmres_res[0].time
            gmres_iters[mt] = gmres_res[0].iterations
    
    # Bar plots
    common_types = list(set(cg_times.keys()) & set(gmres_times.keys()))
    if not common_types:
        return
    
    x = np.arange(len(common_types))
    width = 0.35
    
    # Time comparison - No preconditioner
    axes[0, 0].bar(x - width/2, [cg_times.get(mt, 0) for mt in common_types], 
                   width, label='CG', color='steelblue')
    axes[0, 0].bar(x + width/2, [gmres_times.get(mt, 0) for mt in common_types], 
                   width, label='GMRES', color='coral')
    axes[0, 0].set_xlabel('Matrix Type')
    axes[0, 0].set_ylabel('Time (seconds)')
    axes[0, 0].set_title('Execution Time (No Preconditioner)')
    axes[0, 0].set_xticks(x)
    axes[0, 0].set_xticklabels([mt[:20] for mt in common_types], rotation=45, ha='right')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3, axis='y')
    
    # Iterations comparison - No preconditioner
    axes[0, 1].bar(x - width/2, [cg_iters.get(mt, 0) for mt in common_types], 
                   width, label='CG', color='steelblue')
    axes[0, 1].bar(x + width/2, [gmres_iters.get(mt, 0) for mt in common_types], 
                   width, label='GMRES', color='coral')
    axes[0, 1].set_xlabel('Matrix Type')
    axes[0, 1].set_ylabel('Iterations')
    axes[0, 1].set_title('Iterations (No Preconditioner)')
    axes[0, 1].set_xticks(x)
    axes[0, 1].set_xticklabels([mt[:20] for mt in common_types], rotation=45, ha='right')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3, axis='y')
    
    # With ILU preconditioner
    ilu_results = [r for r in results if r.preconditioner == 'ILU']
    
    cg_times_ilu = {}
    gmres_times_ilu = {}
    cg_iters_ilu = {}
    gmres_iters_ilu = {}
    
    for mt in matrix_types:
        mt_results = [r for r in ilu_results if r.matrix_type == mt]
        cg_res = [r for r in mt_results if r.method == 'CG']
        gmres_res = [r for r in mt_results if r.method == 'GMRES']
        
        if cg_res:
            cg_times_ilu[mt] = cg_res[0].time
            cg_iters_ilu[mt] = cg_res[0].iterations
        if gmres_res:
            gmres_times_ilu[mt] = gmres_res[0].time
            gmres_iters_ilu[mt] = gmres_res[0].iterations
    
    common_types_ilu = list(set(cg_times_ilu.keys()) & set(gmres_times_ilu.keys()))
    
    if common_types_ilu:
        x_ilu = np.arange(len(common_types_ilu))
        
        axes[1, 0].bar(x_ilu - width/2, [cg_times_ilu.get(mt, 0) for mt in common_types_ilu], 
                       width, label='CG', color='steelblue')
        axes[1, 0].bar(x_ilu + width/2, [gmres_times_ilu.get(mt, 0) for mt in common_types_ilu], 
                       width, label='GMRES', color='coral')
        axes[1, 0].set_xlabel('Matrix Type')
        axes[1, 0].set_ylabel('Time (seconds)')
        axes[1, 0].set_title('Execution Time (ILU Preconditioner)')
        axes[1, 0].set_xticks(x_ilu)
        axes[1, 0].set_xticklabels([mt[:20] for mt in common_types_ilu], rotation=45, ha='right')
        axes[1, 0].legend()
        axes[1, 0].grid(True, alpha=0.3, axis='y')
        
        axes[1, 1].bar(x_ilu - width/2, [cg_iters_ilu.get(mt, 0) for mt in common_types_ilu], 
                       width, label='CG', color='steelblue')
        axes[1, 1].bar(x_ilu + width/2, [gmres_iters_ilu.get(mt, 0) for mt in common_types_ilu], 
                       width, label='GMRES', color='coral')
        axes[1, 1].set_xlabel('Matrix Type')
        axes[1, 1].set_ylabel('Iterations')
        axes[1, 1].set_title('Iterations (ILU Preconditioner)')
        axes[1, 1].set_xticks(x_ilu)
        axes[1, 1].set_xticklabels([mt[:20] for mt in common_types_ilu], rotation=45, ha='right')
        axes[1, 1].legend()
        axes[1, 1].grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'matrix_type_comparison.png'), dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Saved: matrix_type_comparison.png")


def create_amg_scaling_plot(results: List[SolverResult]):
    """Create AMG scaling comparison plot"""
    
    amg_results = [r for r in results if 'AMG' in r.preconditioner]
    none_results = [r for r in results if r.preconditioner == 'None' and '2D Laplacian' in r.matrix_type]
    
    if not amg_results:
        print("No AMG results available for scaling plot")
        return
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # CG comparison
    cg_none = [r for r in none_results if r.method == 'CG']
    cg_amg = [r for r in amg_results if r.method == 'CG']
    
    cg_none.sort(key=lambda x: x.matrix_size)
    cg_amg.sort(key=lambda x: x.matrix_size)
    
    if cg_none:
        sizes_none = [r.matrix_size for r in cg_none]
        iters_none = [r.iterations for r in cg_none]
        times_none = [r.time for r in cg_none]
        axes[0].plot(sizes_none, iters_none, 'o-', label='CG (No Prec)', color='steelblue')
        axes[1].plot(sizes_none, times_none, 'o-', label='CG (No Prec)', color='steelblue')
    
    if cg_amg:
        sizes_amg = [r.matrix_size for r in cg_amg]
        iters_amg = [r.iterations for r in cg_amg]
        times_amg = [r.time for r in cg_amg]
        axes[0].plot(sizes_amg, iters_amg, 's-', label='CG + AMG', color='forestgreen')
        axes[1].plot(sizes_amg, times_amg, 's-', label='CG + AMG', color='forestgreen')
    
    # GMRES comparison
    gmres_none = [r for r in none_results if r.method == 'GMRES']
    gmres_amg = [r for r in amg_results if r.method == 'GMRES']
    
    gmres_none.sort(key=lambda x: x.matrix_size)
    gmres_amg.sort(key=lambda x: x.matrix_size)
    
    if gmres_none:
        sizes_none = [r.matrix_size for r in gmres_none]
        iters_none = [r.iterations for r in gmres_none]
        times_none = [r.time for r in gmres_none]
        axes[0].plot(sizes_none, iters_none, '^--', label='GMRES (No Prec)', color='coral')
        axes[1].plot(sizes_none, times_none, '^--', label='GMRES (No Prec)', color='coral')
    
    if gmres_amg:
        sizes_amg = [r.matrix_size for r in gmres_amg]
        iters_amg = [r.iterations for r in gmres_amg]
        times_amg = [r.time for r in gmres_amg]
        axes[0].plot(sizes_amg, iters_amg, 'd--', label='GMRES + AMG', color='darkred')
        axes[1].plot(sizes_amg, times_amg, 'd--', label='GMRES + AMG', color='darkred')
    
    axes[0].set_xlabel('Matrix Size')
    axes[0].set_ylabel('Iterations')
    axes[0].set_title('AMG Preconditioning: Iteration Scaling')
    axes[0].legend()
    axes[0].set_xscale('log')
    axes[0].grid(True, alpha=0.3)
    
    axes[1].set_xlabel('Matrix Size')
    axes[1].set_ylabel('Time (seconds)')
    axes[1].set_title('AMG Preconditioning: Time Scaling')
    axes[1].legend()
    axes[1].set_xscale('log')
    axes[1].set_yscale('log')
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'amg_scaling.png'), dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Saved: amg_scaling.png")


def create_ill_conditioned_comparison(results: List[SolverResult]):
    """Create comparison for ill-conditioned systems"""
    
    aniso_results = [r for r in results if 'Anisotropic' in r.matrix_type]
    
    if not aniso_results:
        return
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 12))
    
    # Extract epsilon values
    eps_values = []
    for r in aniso_results:
        if 'eps=' in r.matrix_type:
            try:
                eps = float(r.matrix_type.split('eps=')[1].replace(')', ''))
                eps_values.append(eps)
            except:
                pass
    
    eps_values = sorted(list(set(eps_values)), reverse=True)
    
    preconditioners = list(set(r.preconditioner for r in aniso_results))
    
    for prec in preconditioners:
        prec_results = [r for r in aniso_results if r.preconditioner == prec]
        
        cg_results = [r for r in prec_results if r.method == 'CG']
        gmres_results = [r for r in prec_results if r.method == 'GMRES']
        
        # Extract data points
        cg_data = {}
        gmres_data = {}
        
        for r in cg_results:
            if 'eps=' in r.matrix_type:
                try:
                    eps = float(r.matrix_type.split('eps=')[1].replace(')', ''))
                    cg_data[eps] = (r.iterations, r.time, r.converged)
                except:
                    pass
        
        for r in gmres_results:
            if 'eps=' in r.matrix_type:
                try:
                    eps = float(r.matrix_type.split('eps=')[1].replace(')', ''))
                    gmres_data[eps] = (r.iterations, r.time, r.converged)
                except:
                    pass
        
        if cg_data:
            eps_sorted = sorted(cg_data.keys(), reverse=True)
            iters = [cg_data[e][0] for e in eps_sorted]
            times = [cg_data[e][1] for e in eps_sorted]
            
            axes[0, 0].plot(eps_sorted, iters, 'o-', label=f'CG ({prec})')
            axes[0, 1].plot(eps_sorted, times, 'o-', label=f'CG ({prec})')
        
        if gmres_data:
            eps_sorted = sorted(gmres_data.keys(), reverse=True)
            iters = [gmres_data[e][0] for e in eps_sorted]
            times = [gmres_data[e][1] for e in eps_sorted]
            
            axes[1, 0].plot(eps_sorted, iters, 's--', label=f'GMRES ({prec})')
            axes[1, 1].plot(eps_sorted, times, 's--', label=f'GMRES ({prec})')
    
    for ax in axes.flat:
        ax.set_xscale('log')
        ax.invert_xaxis()
        ax.grid(True, alpha=0.3)
    
    axes[0, 0].set_xlabel('Anisotropy ε (log scale, decreasing →)')
    axes[0, 0].set_ylabel('Iterations')
    axes[0, 0].set_title('CG: Iterations vs Anisotropy')
    axes[0, 0].legend()
    
    axes[0, 1].set_xlabel('Anisotropy ε (log scale, decreasing →)')
    axes[0, 1].set_ylabel('Time (seconds)')
    axes[0, 1].set_title('CG: Time vs Anisotropy')
    axes[0, 1].legend()
    axes[0, 1].set_yscale('log')
    
    axes[1, 0].set_xlabel('Anisotropy ε (log scale, decreasing →)')
    axes[1, 0].set_ylabel('Iterations')
    axes[1, 0].set_title('GMRES: Iterations vs Anisotropy')
    axes[1, 0].legend()
    
    axes[1, 1].set_xlabel('Anisotropy ε (log scale, decreasing →)')
    axes[1, 1].set_ylabel('Time (seconds)')
    axes[1, 1].set_title('GMRES: Time vs Anisotropy')
    axes[1, 1].legend()
    axes[1, 1].set_yscale('log')
    
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'ill_conditioned_comparison.png'), dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Saved: ill_conditioned_comparison.png")


def create_cg_vs_gmres_direct_comparison(results: List[SolverResult]):
    """Create direct CG vs GMRES scatter plot comparison"""
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Pair up CG and GMRES results
    # Group by (matrix_type, matrix_size, preconditioner)
    groups = defaultdict(list)
    
    for r in results:
        key = (r.matrix_type, r.matrix_size, r.preconditioner)
        groups[key].append(r)
    
    cg_times = []
    gmres_times = []
    cg_iters = []
    gmres_iters = []
    labels = []
    
    for key, group_results in groups.items():
        cg_res = [r for r in group_results if r.method == 'CG']
        gmres_res = [r for r in group_results if r.method == 'GMRES']
        
        if cg_res and gmres_res:
            cg_times.append(cg_res[0].time)
            gmres_times.append(gmres_res[0].time)
            cg_iters.append(cg_res[0].iterations)
            gmres_iters.append(gmres_res[0].iterations)
            labels.append(f"{key[2]}")
    
    if not cg_times:
        return
    
    # Color by preconditioner type
    unique_precs = list(set(labels))
    colors = plt.cm.tab10(np.linspace(0, 1, len(unique_precs)))
    color_map = {p: colors[i] for i, p in enumerate(unique_precs)}
    point_colors = [color_map[l] for l in labels]
    
    # Time scatter plot
    axes[0].scatter(cg_times, gmres_times, c=point_colors, alpha=0.7, s=50)
    
    # Add diagonal line
    max_time = max(max(cg_times), max(gmres_times))
    axes[0].plot([0, max_time], [0, max_time], 'k--', alpha=0.5, label='CG = GMRES')
    
    axes[0].set_xlabel('CG Time (seconds)')
    axes[0].set_ylabel('GMRES Time (seconds)')
    axes[0].set_title('Execution Time: CG vs GMRES')
    axes[0].grid(True, alpha=0.3)
    axes[0].set_yscale('log')
    
    # Add legend for preconditioners
    for prec in unique_precs:
        axes[0].scatter([], [], c=[color_map[prec]], label=prec, s=50)
    axes[0].legend(title='Preconditioner', loc='upper left')
    
    # Iterations scatter plot
    axes[1].scatter(cg_iters, gmres_iters, c=point_colors, alpha=0.7, s=50)
    
    # Add diagonal line
    max_iters = max(max(cg_iters), max(gmres_iters))
    axes[1].plot([0, max_iters], [0, max_iters], 'k--', alpha=0.5, label='CG = GMRES')
    
    axes[1].set_xlabel('CG Iterations')
    axes[1].set_ylabel('GMRES Iterations')
    axes[1].set_title('Iteration Count: CG vs GMRES')
    axes[1].grid(True, alpha=0.3)
    
    # Add legend for preconditioners
    for prec in unique_precs:
        axes[1].scatter([], [], c=[color_map[prec]], label=prec, s=50)
    axes[1].legend(title='Preconditioner', loc='upper left')
    
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'cg_vs_gmres_scatter.png'), dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Saved: cg_vs_gmres_scatter.png")


def create_speedup_heatmap(results: List[SolverResult]):
    """Create heatmap showing CG speedup over GMRES"""
    
    # Group results
    groups = defaultdict(list)
    
    for r in results:
        key = (r.matrix_type, r.preconditioner)
        groups[key].append(r)
    
    matrix_types = sorted(list(set(r.matrix_type for r in results)))
    preconditioners = sorted(list(set(r.preconditioner for r in results)))
    
    # Create speedup matrix (GMRES_time / CG_time)
    speedup_time = np.zeros((len(matrix_types), len(preconditioners)))
    speedup_iters = np.zeros((len(matrix_types), len(preconditioners)))
    
    for i, mt in enumerate(matrix_types):
        for j, prec in enumerate(preconditioners):
            key = (mt, prec)
            if key in groups:
                group_results = groups[key]
                cg_res = [r for r in group_results if r.method == 'CG']
                gmres_res = [r for r in group_results if r.method == 'GMRES']
                
                if cg_res and gmres_res:
                    # Speedup > 1 means CG is faster
                    speedup_time[i, j] = gmres_res[0].time / max(cg_res[0].time, 1e-10)
                    speedup_iters[i, j] = gmres_res[0].iterations / max(cg_res[0].iterations, 1)
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    
    # Time speedup heatmap
    im1 = axes[0].imshow(speedup_time, cmap='RdYlGn', aspect='auto', 
                          vmin=0.5, vmax=2.0)
    axes[0].set_xticks(np.arange(len(preconditioners)))
    axes[0].set_yticks(np.arange(len(matrix_types)))
    axes[0].set_xticklabels(preconditioners, rotation=45, ha='right')
    axes[0].set_yticklabels([mt[:25] for mt in matrix_types])
    axes[0].set_title('Time Speedup (GMRES/CG)\n>1 = CG faster, <1 = GMRES faster')
    
    # Add text annotations
    for i in range(len(matrix_types)):
        for j in range(len(preconditioners)):
            if speedup_time[i, j] > 0:
                text = axes[0].text(j, i, f'{speedup_time[i, j]:.2f}',
                                   ha='center', va='center', fontsize=8)
    
    plt.colorbar(im1, ax=axes[0])
    
    # Iteration speedup heatmap
    im2 = axes[1].imshow(speedup_iters, cmap='RdYlGn', aspect='auto',
                          vmin=0.5, vmax=2.0)
    axes[1].set_xticks(np.arange(len(preconditioners)))
    axes[1].set_yticks(np.arange(len(matrix_types)))
    axes[1].set_xticklabels(preconditioners, rotation=45, ha='right')
    axes[1].set_yticklabels([mt[:25] for mt in matrix_types])
    axes[1].set_title('Iteration Ratio (GMRES/CG)\n>1 = CG fewer iters, <1 = GMRES fewer iters')
    
    # Add text annotations
    for i in range(len(matrix_types)):
        for j in range(len(preconditioners)):
            if speedup_iters[i, j] > 0:
                text = axes[1].text(j, i, f'{speedup_iters[i, j]:.2f}',
                                   ha='center', va='center', fontsize=8)
    
    plt.colorbar(im2, ax=axes[1])
    
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'speedup_heatmap.png'), dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Saved: speedup_heatmap.png")


def create_convergence_comparison(results: List[SolverResult]):
    """Create convergence rate comparison plots"""
    
    # We need to run specific cases with residual history
    print("\nGenerating convergence history plots...")
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    
    # Test cases
    test_cases = [
        ('Well-conditioned (κ≈10)', 1000, 10),
        ('Moderate (κ≈100)', 1000, 100),
        ('Ill-conditioned (κ≈1000)', 1000, 1000),
    ]
    
    preconditioners_to_test = [
        ('None', lambda A: None),
        ('ILU', get_ilu_preconditioner),
    ]
    
    if PYAMG_AVAILABLE:
        preconditioners_to_test.append(('AMG', get_amg_preconditioner))
    
    for col, (case_name, size, cond) in enumerate(test_cases):
        A = generate_modified_laplacian(size, cond)
        np.random.seed(123)
        x_true = np.random.randn(size)
        b = A @ x_true
        
        for prec_name, prec_func in preconditioners_to_test:
            M = prec_func(A)
            
            # CG with residual history
            cg_counter = IterationCounter()
            x_cg, _ = cg(A, b, M=M, rtol=1e-12, maxiter=2000, callback=cg_counter)
            
            # GMRES with residual history
            gmres_counter = IterationCounter()
            x_gmres, _ = gmres(A, b, M=M, rtol=1e-12, maxiter=2000, 
                               callback=gmres_counter, callback_type='pr_norm')
            
            # Plot CG convergence
            if cg_counter.residuals:
                axes[0, col].semilogy(cg_counter.residuals, label=f'CG ({prec_name})')
            
            # Plot GMRES convergence
            if gmres_counter.residuals:
                axes[1, col].semilogy(gmres_counter.residuals, label=f'GMRES ({prec_name})')
        
        axes[0, col].set_xlabel('Iteration')
        axes[0, col].set_ylabel('Residual Norm')
        axes[0, col].set_title(f'CG Convergence\n{case_name}')
        axes[0, col].legend()
        axes[0, col].grid(True, alpha=0.3)
        
        axes[1, col].set_xlabel('Iteration')
        axes[1, col].set_ylabel('Residual Norm')
        axes[1, col].set_title(f'GMRES Convergence\n{case_name}')
        axes[1, col].legend()
        axes[1, col].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'convergence_comparison.png'), dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Saved: convergence_comparison.png")


def create_summary_table(results: List[SolverResult]):
    """Create and save a summary table of all results"""
    
    # Create summary dataframe-like structure
    summary_lines = []
    summary_lines.append("=" * 120)
    summary_lines.append("COMPREHENSIVE CG vs GMRES COMPARISON - SUMMARY TABLE")
    summary_lines.append("=" * 120)
    summary_lines.append("")
    
    header = f"{'Matrix Type':<30} {'Size':>8} {'Prec':<12} {'CG Iter':>10} {'CG Time':>12} {'GMRES Iter':>10} {'GMRES Time':>12} {'Speedup':>10}"
    summary_lines.append(header)
    summary_lines.append("-" * 120)
    
    # Group results
    groups = defaultdict(list)
    for r in results:
        key = (r.matrix_type, r.matrix_size, r.preconditioner)
        groups[key].append(r)
    
    for key in sorted(groups.keys()):
        group_results = groups[key]
        cg_res = [r for r in group_results if r.method == 'CG']
        gmres_res = [r for r in group_results if r.method == 'GMRES']
        
        if cg_res and gmres_res:
            speedup = gmres_res[0].time / max(cg_res[0].time, 1e-10)
            line = f"{key[0][:30]:<30} {key[1]:>8} {key[2]:<12} {cg_res[0].iterations:>10} {cg_res[0].time:>12.6f} {gmres_res[0].iterations:>10} {gmres_res[0].time:>12.6f} {speedup:>10.2f}x"
            summary_lines.append(line)
    
    summary_lines.append("")
    summary_lines.append("=" * 120)
    summary_lines.append("NOTES:")
    summary_lines.append("- Speedup > 1.0 indicates CG is faster than GMRES")
    summary_lines.append("- Speedup < 1.0 indicates GMRES is faster than CG")
    summary_lines.append("- For SPD matrices, CG is theoretically optimal")
    summary_lines.append("- GMRES has higher memory requirements but more general applicability")
    summary_lines.append("=" * 120)
    
    summary_text = "\n".join(summary_lines)
    
    # Save to file
    with open(os.path.join(OUTPUT_DIR, 'summary_table.txt'), 'w') as f:
        f.write(summary_text)
    
    print(f"Saved: summary_table.txt")
    
    # Also print to console
    print("\n" + summary_text)


def create_performance_profile(results: List[SolverResult]):
    """Create performance profile plots"""
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Group results and compute performance ratios
    groups = defaultdict(list)
    for r in results:
        key = (r.matrix_type, r.matrix_size, r.preconditioner)
        groups[key].append(r)
    
    cg_ratios_time = []
    gmres_ratios_time = []
    cg_ratios_iter = []
    gmres_ratios_iter = []
    
    for key, group_results in groups.items():
        cg_res = [r for r in group_results if r.method == 'CG']
        gmres_res = [r for r in group_results if r.method == 'GMRES']
        
        if cg_res and gmres_res:
            min_time = min(cg_res[0].time, gmres_res[0].time)
            min_iter = min(cg_res[0].iterations, gmres_res[0].iterations)
            
            if min_time > 0:
                cg_ratios_time.append(cg_res[0].time / min_time)
                gmres_ratios_time.append(gmres_res[0].time / min_time)
            
            if min_iter > 0:
                cg_ratios_iter.append(cg_res[0].iterations / min_iter)
                gmres_ratios_iter.append(gmres_res[0].iterations / min_iter)
    
    # Sort ratios
    cg_ratios_time.sort()
    gmres_ratios_time.sort()
    cg_ratios_iter.sort()
    gmres_ratios_iter.sort()
    
    # Compute performance profile
    def compute_profile(ratios, tau_max=10):
        taus = np.linspace(1, tau_max, 100)
        profile = np.zeros_like(taus)
        n = len(ratios)
        for i, tau in enumerate(taus):
            profile[i] = sum(r <= tau for r in ratios) / n
        return taus, profile
    
    if cg_ratios_time and gmres_ratios_time:
        taus, cg_profile_time = compute_profile(cg_ratios_time)
        _, gmres_profile_time = compute_profile(gmres_ratios_time)
        
        axes[0].plot(taus, cg_profile_time, 'b-', linewidth=2, label='CG')
        axes[0].plot(taus, gmres_profile_time, 'r--', linewidth=2, label='GMRES')
        axes[0].set_xlabel('Performance Ratio τ')
        axes[0].set_ylabel('Fraction of Problems Solved')
        axes[0].set_title('Performance Profile (Execution Time)')
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)
        axes[0].set_xlim([1, 10])
        axes[0].set_ylim([0, 1.05])
    
    if cg_ratios_iter and gmres_ratios_iter:
        taus, cg_profile_iter = compute_profile(cg_ratios_iter)
        _, gmres_profile_iter = compute_profile(gmres_ratios_iter)
        
        axes[1].plot(taus, cg_profile_iter, 'b-', linewidth=2, label='CG')
        axes[1].plot(taus, gmres_profile_iter, 'r--', linewidth=2, label='GMRES')
        axes[1].set_xlabel('Performance Ratio τ')
        axes[1].set_ylabel('Fraction of Problems Solved')
        axes[1].set_title('Performance Profile (Iteration Count)')
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)
        axes[1].set_xlim([1, 10])
        axes[1].set_ylim([0, 1.05])
    
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'performance_profile.png'), dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Saved: performance_profile.png")


def create_3d_surface_plots(results: List[SolverResult]):
    """Create 3D surface plots for time/iterations vs size and condition number"""
    
    from mpl_toolkits.mplot3d import Axes3D
    
    # Filter for modified Laplacian results
    mod_laplacian_results = [r for r in results if 'Modified Laplacian' in r.matrix_type]
    
    if not mod_laplacian_results:
        return
    
    fig = plt.figure(figsize=(16, 12))
    
    # CG - No preconditioner
    cg_none = [r for r in mod_laplacian_results 
               if r.method == 'CG' and r.preconditioner == 'None']
    
    if cg_none:
        sizes = [r.matrix_size for r in cg_none]
        conds = [r.condition_number for r in cg_none]
        iters = [r.iterations for r in cg_none]
        times = [r.time for r in cg_none]
        
        ax1 = fig.add_subplot(2, 2, 1, projection='3d')
        ax1.scatter(np.log10(sizes), np.log10(conds), iters, c='steelblue', s=50)
        ax1.set_xlabel('log10(Size)')
        ax1.set_ylabel('log10(Condition Number)')
        ax1.set_zlabel('Iterations')
        ax1.set_title('CG Iterations (No Preconditioner)')
    
    # GMRES - No preconditioner
    gmres_none = [r for r in mod_laplacian_results 
                  if r.method == 'GMRES' and r.preconditioner == 'None']
    
    if gmres_none:
        sizes = [r.matrix_size for r in gmres_none]
        conds = [r.condition_number for r in gmres_none]
        iters = [r.iterations for r in gmres_none]
        
        ax2 = fig.add_subplot(2, 2, 2, projection='3d')
        ax2.scatter(np.log10(sizes), np.log10(conds), iters, c='coral', s=50)
        ax2.set_xlabel('log10(Size)')
        ax2.set_ylabel('log10(Condition Number)')
        ax2.set_zlabel('Iterations')
        ax2.set_title('GMRES Iterations (No Preconditioner)')
    
    # CG - ILU preconditioner
    cg_ilu = [r for r in mod_laplacian_results 
              if r.method == 'CG' and r.preconditioner == 'ILU']
    
    if cg_ilu:
        sizes = [r.matrix_size for r in cg_ilu]
        conds = [r.condition_number for r in cg_ilu]
        iters = [r.iterations for r in cg_ilu]
        
        ax3 = fig.add_subplot(2, 2, 3, projection='3d')
        ax3.scatter(np.log10(sizes), np.log10(conds), iters, c='forestgreen', s=50)
        ax3.set_xlabel('log10(Size)')
        ax3.set_ylabel('log10(Condition Number)')
        ax3.set_zlabel('Iterations')
        ax3.set_title('CG Iterations (ILU Preconditioner)')
    
    # GMRES - ILU preconditioner
    gmres_ilu = [r for r in mod_laplacian_results 
                 if r.method == 'GMRES' and r.preconditioner == 'ILU']
    
    if gmres_ilu:
        sizes = [r.matrix_size for r in gmres_ilu]
        conds = [r.condition_number for r in gmres_ilu]
        iters = [r.iterations for r in gmres_ilu]
        
        ax4 = fig.add_subplot(2, 2, 4, projection='3d')
        ax4.scatter(np.log10(sizes), np.log10(conds), iters, c='darkred', s=50)
        ax4.set_xlabel('log10(Size)')
        ax4.set_ylabel('log10(Condition Number)')
        ax4.set_zlabel('Iterations')
        ax4.set_title('GMRES Iterations (ILU Preconditioner)')
    
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, '3d_surface_plots.png'), dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Saved: 3d_surface_plots.png")


def create_efficiency_plots(results: List[SolverResult]):
    """Create efficiency comparison plots (time per iteration, memory estimates)"""
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 12))
    
    # Group by size
    laplacian_results = [r for r in results if '2D Laplacian' in r.matrix_type]
    
    sizes = sorted(list(set(r.matrix_size for r in laplacian_results)))
    
    # Time per iteration
    for prec in ['None', 'ILU']:
        prec_results = [r for r in laplacian_results if r.preconditioner == prec]
        
        cg_results = [r for r in prec_results if r.method == 'CG']
        gmres_results = [r for r in prec_results if r.method == 'GMRES']
        
        cg_results.sort(key=lambda x: x.matrix_size)
        gmres_results.sort(key=lambda x: x.matrix_size)
        
        if cg_results:
            sizes_cg = [r.matrix_size for r in cg_results]
            time_per_iter_cg = [r.time / max(r.iterations, 1) * 1000 for r in cg_results]  # ms
            axes[0, 0].plot(sizes_cg, time_per_iter_cg, 'o-', label=f'CG ({prec})')
        
        if gmres_results:
            sizes_gmres = [r.matrix_size for r in gmres_results]
            time_per_iter_gmres = [r.time / max(r.iterations, 1) * 1000 for r in gmres_results]
            axes[0, 0].plot(sizes_gmres, time_per_iter_gmres, 's--', label=f'GMRES ({prec})')
    
    axes[0, 0].set_xlabel('Matrix Size')
    axes[0, 0].set_ylabel('Time per Iteration (ms)')
    axes[0, 0].set_title('Computational Cost per Iteration')
    axes[0, 0].legend()
    axes[0, 0].set_xscale('log')
    axes[0, 0].set_yscale('log')
    axes[0, 0].grid(True, alpha=0.3)
    
    # Total work comparison (iterations * size as proxy)
    for prec in ['None', 'ILU']:
        prec_results = [r for r in laplacian_results if r.preconditioner == prec]
        
        cg_results = [r for r in prec_results if r.method == 'CG']
        gmres_results = [r for r in prec_results if r.method == 'GMRES']
        
        cg_results.sort(key=lambda x: x.matrix_size)
        gmres_results.sort(key=lambda x: x.matrix_size)
        
        if cg_results:
            sizes_cg = [r.matrix_size for r in cg_results]
            work_cg = [r.iterations * r.matrix_size for r in cg_results]
            axes[0, 1].plot(sizes_cg, work_cg, 'o-', label=f'CG ({prec})')
        
        if gmres_results:
            sizes_gmres = [r.matrix_size for r in gmres_results]
            work_gmres = [r.iterations * r.matrix_size for r in gmres_results]
            axes[0, 1].plot(sizes_gmres, work_gmres, 's--', label=f'GMRES ({prec})')
    
    axes[0, 1].set_xlabel('Matrix Size')
    axes[0, 1].set_ylabel('Total Work (iterations × size)')
    axes[0, 1].set_title('Total Computational Work')
    axes[0, 1].legend()
    axes[0, 1].set_xscale('log')
    axes[0, 1].set_yscale('log')
    axes[0, 1].grid(True, alpha=0.3)
    
    # Estimated memory usage (GMRES uses more due to Krylov basis storage)
    if sizes:
        restart = 50  # GMRES restart parameter
        
        # CG: O(n) - stores only a few vectors
        cg_memory = [4 * n * 8 / 1e6 for n in sizes]  # 4 vectors, 8 bytes per float, MB
        
        # GMRES: O(n * restart) - stores Krylov basis
        gmres_memory = [(restart + 3) * n * 8 / 1e6 for n in sizes]
        
        axes[1, 0].plot(sizes, cg_memory, 'o-', label='CG', color='steelblue')
        axes[1, 0].plot(sizes, gmres_memory, 's--', label=f'GMRES (restart={restart})', color='coral')
        axes[1, 0].set_xlabel('Matrix Size')
        axes[1, 0].set_ylabel('Estimated Memory (MB)')
        axes[1, 0].set_title('Memory Requirements (Vector Storage Only)')
        axes[1, 0].legend()
        axes[1, 0].set_xscale('log')
        axes[1, 0].set_yscale('log')
        axes[1, 0].grid(True, alpha=0.3)
    
    # Iteration efficiency: iterations / sqrt(condition number)
    # For well-conditioned matrices, both should be similar
    cond_results = [r for r in results if 'Modified Laplacian' in r.matrix_type]
    
    for prec in ['None', 'ILU']:
        prec_results = [r for r in cond_results if r.preconditioner == prec]
        
        cg_results = [r for r in prec_results if r.method == 'CG']
        gmres_results = [r for r in prec_results if r.method == 'GMRES']
        
        cg_results.sort(key=lambda x: x.condition_number)
        gmres_results.sort(key=lambda x: x.condition_number)
        
        if cg_results:
            conds = [r.condition_number for r in cg_results]
            efficiency = [r.iterations / np.sqrt(r.condition_number) for r in cg_results]
            axes[1, 1].plot(conds, efficiency, 'o-', label=f'CG ({prec})')
        
        if gmres_results:
            conds = [r.condition_number for r in gmres_results]
            efficiency = [r.iterations / np.sqrt(r.condition_number) for r in gmres_results]
            axes[1, 1].plot(conds, efficiency, 's--', label=f'GMRES ({prec})')
    
    axes[1, 1].set_xlabel('Condition Number')
    axes[1, 1].set_ylabel('Iterations / √(Condition Number)')
    axes[1, 1].set_title('Iteration Efficiency (CG theory: O(√κ))')
    axes[1, 1].legend()
    axes[1, 1].set_xscale('log')
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'efficiency_comparison.png'), dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Saved: efficiency_comparison.png")


def create_robustness_plot(results: List[SolverResult]):
    """Create robustness comparison showing convergence success rates"""
    
    # Group by preconditioner and matrix type
    matrix_types = list(set(r.matrix_type for r in results))
    preconditioners = list(set(r.preconditioner for r in results))
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Convergence rates by preconditioner
    cg_conv_rates = {}
    gmres_conv_rates = {}
    
    for prec in preconditioners:
        prec_results = [r for r in results if r.preconditioner == prec]
        cg_results = [r for r in prec_results if r.method == 'CG']
        gmres_results = [r for r in prec_results if r.method == 'GMRES']
        
        if cg_results:
            cg_conv_rates[prec] = sum(r.converged for r in cg_results) / len(cg_results) * 100
        if gmres_results:
            gmres_conv_rates[prec] = sum(r.converged for r in gmres_results) / len(gmres_results) * 100
    
    common_precs = sorted(list(set(cg_conv_rates.keys()) & set(gmres_conv_rates.keys())))
    
    if common_precs:
        x = np.arange(len(common_precs))
        width = 0.35
        
        cg_rates = [cg_conv_rates.get(p, 0) for p in common_precs]
        gmres_rates = [gmres_conv_rates.get(p, 0) for p in common_precs]
        
        axes[0].bar(x - width/2, cg_rates, width, label='CG', color='steelblue')
        axes[0].bar(x + width/2, gmres_rates, width, label='GMRES', color='coral')
        axes[0].set_xlabel('Preconditioner')
        axes[0].set_ylabel('Convergence Rate (%)')
        axes[0].set_title('Convergence Success Rate by Preconditioner')
        axes[0].set_xticks(x)
        axes[0].set_xticklabels(common_precs, rotation=45, ha='right')
        axes[0].legend()
        axes[0].set_ylim([0, 105])
        axes[0].grid(True, alpha=0.3, axis='y')
    
    # Average iterations to convergence by preconditioner
    cg_avg_iters = {}
    gmres_avg_iters = {}
    
    for prec in preconditioners:
        prec_results = [r for r in results if r.preconditioner == prec]
        cg_results = [r for r in prec_results if r.method == 'CG' and r.converged]
        gmres_results = [r for r in prec_results if r.method == 'GMRES' and r.converged]
        
        if cg_results:
            cg_avg_iters[prec] = np.mean([r.iterations for r in cg_results])
        if gmres_results:
            gmres_avg_iters[prec] = np.mean([r.iterations for r in gmres_results])
    
    common_precs = sorted(list(set(cg_avg_iters.keys()) & set(gmres_avg_iters.keys())))
    
    if common_precs:
        x = np.arange(len(common_precs))
        width = 0.35
        
        cg_iters = [cg_avg_iters.get(p, 0) for p in common_precs]
        gmres_iters = [gmres_avg_iters.get(p, 0) for p in common_precs]
        
        axes[1].bar(x - width/2, cg_iters, width, label='CG', color='steelblue')
        axes[1].bar(x + width/2, gmres_iters, width, label='GMRES', color='coral')
        axes[1].set_xlabel('Preconditioner')
        axes[1].set_ylabel('Average Iterations')
        axes[1].set_title('Average Iterations to Convergence')
        axes[1].set_xticks(x)
        axes[1].set_xticklabels(common_precs, rotation=45, ha='right')
        axes[1].legend()
        axes[1].grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'robustness_comparison.png'), dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Saved: robustness_comparison.png")


def save_results_to_json(results: List[SolverResult]):
    """Save all results to JSON for further analysis"""
    
    results_dict = []
    for r in results:
        results_dict.append({
            'method': r.method,
            'matrix_type': r.matrix_type,
            'matrix_size': r.matrix_size,
            'condition_number': float(r.condition_number) if not np.isnan(r.condition_number) else None,
            'preconditioner': r.preconditioner,
            'time': r.time,
            'iterations': r.iterations,
            'converged': r.converged,
            'residual_norm': float(r.residual_norm) if not np.isnan(r.residual_norm) else None,
        })
    
    with open(os.path.join(OUTPUT_DIR, 'results.json'), 'w') as f:
        json.dump(results_dict, f, indent=2)
    
    print(f"Saved: results.json")


# =============================================================================
# Main Execution
# =============================================================================

def main():
    """Main function to run all comparisons and generate plots"""
    
    print(f"\nOutput directory: {OUTPUT_DIR}")
    print(f"pyamg available: {PYAMG_AVAILABLE}")
    
    # Run comprehensive comparison
    results = run_comprehensive_comparison()
    
    print("\n" + "=" * 60)
    print("GENERATING VISUALIZATION PLOTS")
    print("=" * 60)
    
    # Generate all plots
    print("\nCreating plots...")
    
    try:
        create_size_comparison_plots(results)
    except Exception as e:
        print(f"Error in size_comparison_plots: {e}")
    
    try:
        create_condition_number_plots(results)
    except Exception as e:
        print(f"Error in condition_number_plots: {e}")
    
    try:
        create_preconditioner_comparison_plots(results)
    except Exception as e:
        print(f"Error in preconditioner_comparison_plots: {e}")
    
    try:
        create_matrix_type_comparison_plots(results)
    except Exception as e:
        print(f"Error in matrix_type_comparison_plots: {e}")
    
    try:
        create_amg_scaling_plot(results)
    except Exception as e:
        print(f"Error in amg_scaling_plot: {e}")
    
    try:
        create_ill_conditioned_comparison(results)
    except Exception as e:
        print(f"Error in ill_conditioned_comparison: {e}")
    
    try:
        create_cg_vs_gmres_direct_comparison(results)
    except Exception as e:
        print(f"Error in cg_vs_gmres_direct_comparison: {e}")
    
    try:
        create_speedup_heatmap(results)
    except Exception as e:
        print(f"Error in speedup_heatmap: {e}")
    
    try:
        create_convergence_comparison(results)
    except Exception as e:
        print(f"Error in convergence_comparison: {e}")
    
    try:
        create_performance_profile(results)
    except Exception as e:
        print(f"Error in performance_profile: {e}")
    
    try:
        create_3d_surface_plots(results)
    except Exception as e:
        print(f"Error in 3d_surface_plots: {e}")
    
    try:
        create_efficiency_plots(results)
    except Exception as e:
        print(f"Error in efficiency_plots: {e}")
    
    try:
        create_robustness_plot(results)
    except Exception as e:
        print(f"Error in robustness_plot: {e}")
    
    # Save results
    try:
        save_results_to_json(results)
    except Exception as e:
        print(f"Error saving JSON: {e}")
    
    try:
        create_summary_table(results)
    except Exception as e:
        print(f"Error in summary_table: {e}")
    
    print("\n" + "=" * 60)
    print("COMPARISON COMPLETE")
    print("=" * 60)
    print(f"\nAll results saved to: {OUTPUT_DIR}")
    print(f"\nGenerated files:")
    for f in os.listdir(OUTPUT_DIR):
        print(f"  - {f}")


if __name__ == "__main__":
    main()

In [ ]:
from ngsolve import *
from ngsolve.la import EigenValues_Preconditioner
# switch on again generation of tables
import netgen.meshing
netgen.meshing.Mesh.EnableTableClass("edges", True)
netgen.meshing.Mesh.EnableTableClass("faces", True)

with TaskManager():
    mesh = Mesh(unit_cube.GenerateMesh(maxh=0.1))
    for l in range(1): mesh.Refine()

fes = HCurl(mesh, order=0)
print ("ndof = ", fes.ndof)
u,v = fes.TnT()

a = BilinearForm(curl(u)*curl(v)*dx + 0.01*u*v*dx)

pre = Preconditioner(a, "hcurlamg")
with TaskManager():
    a.Assemble()
    lam = EigenValues_Preconditioner(a.mat, pre.mat)
    print (list(lam[0:3]), '...', list(lam[-3:-1]))

f = LinearForm(curl(v)[2]*dx).Assemble()
gfu = GridFunction(fes)
from ngsolve.krylovspace import CGSolver

inv = CGSolver(a.mat, pre.mat, plotrates=False, maxiter=200)
gfu.vec[:] = inv*f.vec

from ngsolve import *
from netgen.occ import *
from ngsolve.webgui import Draw, FieldLines

coil = Cylinder(Axes((0,0,-0.4),Z), r=0.4, h=0.8) - \
     Cylinder(Axes((0,0,-0.4),Z), r=0.2, h=0.8)
coil.solids.name="coil"
coil.faces.col=(1,0,0)
coil.faces.maxh=0.04

box = Box((-2,-2,-2), (2,2,2) )
box.faces.col=(0,0,1, 0.5)
box.faces.name="outer"

air = box-coil
air.solids.name="air"

shape = Glue([coil,air])

ea = { "euler_angles" : (-60, 0, -11) }
Draw (shape, **ea);

with TaskManager():
    mesh = Mesh(OCCGeometry(shape).GenerateMesh(maxh=0.3)).Curve(5)
    for l in range(1):
        mesh.Refine()

fes = HCurl(mesh, order=0)
print ("ndof = ", fes.ndof)
u,v = fes.TnT()

mu = 4*pi*1e-7
a = BilinearForm(1/mu*curl(u)*curl(v)*dx + 1e-6/mu*u*v*dx)
pre = preconditioners.HCurlAMG(a, verbose=2, smoothingsteps=3, smoothedprolongation=True, maxcoarse=10,
                              maxlevel=10, potentialsmoother='amg')

with TaskManager():
    a.Assemble()

from ngsolve.la import EigenValues_Preconditioner
with TaskManager():
     lam = EigenValues_Preconditioner(a.mat, pre)
print (list(lam[0:3]), list(lam[-3:]))
tau = CF((y,-x,0))
tau = tau/Norm(tau)

N = 1000 # turns
I = 1  # current
crosssection = 0.2*0.8
j = N*I/crosssection

f = LinearForm(j*tau*v*dx("coil", bonus_intorder=6)).Assemble()

gfu = GridFunction(fes)
with TaskManager():
    inv = solvers.CG(a.mat, f.vec, pre=pre.mat, tol=1e-8, plotrates=True)
    gfu.vec.data = inv*f.vec

ea = { "euler_angles" : (-60, 0, -11) }
clipping = { "clipping" : { "y":1, "z":0, "dist":0.5} }

fieldlines = FieldLines(curl(gfu), mesh.Materials(".*"), length=2, num_lines=100)
Draw(curl(gfu), mesh,  "X", draw_vol=False, draw_surf=True, objects=[fieldlines], \
    min=0, max=5e-4, autoscale=False, settings={"Objects": {"Surface": False, "Wireframe": False}}, **ea);

In [1]:
"""
Test script for multi-domain field reconstruction in NGSolve.

This script demonstrates:
1. Creating a split waveguide geometry (two subdomains)
2. Solving eigenvalue problems on each subdomain independently
3. Correctly reconstructing fields on the global domain WITHOUT overwriting
"""

from ngsolve import *
from ngsolve.webgui import Draw
from netgen.occ import *
import numpy as np

# ============================================================================
# GEOMETRY: Two connected rectangular waveguide sections
# ============================================================================

def create_split_waveguide_geometry(
    length1: float = 1.0,
    length2: float = 1.0,
    width: float = 0.5,
    height: float = 0.3,
    mesh_size: float = 0.1
):
    """
    Create a waveguide split into two domains with a shared interface.
    """
    # Create two boxes
    box1 = Box(Pnt(0, 0, 0), Pnt(length1, width, height))
    box1.name = "domain1"
    box1.mat("domain1")

    box2 = Box(Pnt(length1, 0, 0), Pnt(length1 + length2, width, height))
    box2.name = "domain2"
    box2.mat("domain2")

    # Name faces for boundary conditions
    box1.faces.Min(X).name = "port1"
    box1.faces.Max(X).name = "interface"
    box1.faces.Min(Y).name = "wall"
    box1.faces.Max(Y).name = "wall"
    box1.faces.Min(Z).name = "wall"
    box1.faces.Max(Z).name = "wall"

    box2.faces.Min(X).name = "interface"
    box2.faces.Max(X).name = "port2"
    box2.faces.Min(Y).name = "wall"
    box2.faces.Max(Y).name = "wall"
    box2.faces.Min(Z).name = "wall"
    box2.faces.Max(Z).name = "wall"

    # Glue them together
    shape = Glue([box1, box2])

    # Generate mesh
    geo = OCCGeometry(shape)
    ngmesh = geo.GenerateMesh(maxh=mesh_size)
    mesh = Mesh(ngmesh)

    return mesh


def get_element_material_map(mesh):
    """Build element index -> material name mapping."""
    mat_names = list(mesh.GetMaterials())
    print(f"   Materials in mesh: {mat_names}")

    el_to_mat = {}
    mat_to_els = {m: [] for m in mat_names}

    # el.mat returns the material name directly as a string
    for el in mesh.Elements(VOL):
        el_idx = el.nr
        mat_name = el.mat  # This is already a string
        el_to_mat[el_idx] = mat_name
        if mat_name in mat_to_els:
            mat_to_els[mat_name].append(el_idx)

    for mat_name in mat_names:
        print(f"   {mat_name}: {len(mat_to_els[mat_name])} elements")

    return el_to_mat, mat_names, mat_to_els


def get_domain_elements(mesh, domain_name):
    """Get list of element indices belonging to a domain."""
    elements = []
    for el in mesh.Elements(VOL):
        if el.mat == domain_name:
            elements.append(el.nr)
    return elements


# ============================================================================
# EIGENVALUE SOLVER
# ============================================================================

def solve_maxwell_eigenvalue_on_domain(fes, mesh, domain_name: str, n_modes: int = 5):
    """
    Solve Maxwell eigenvalue problem restricted to a domain.
    """
    u, v = fes.TnT()

    # Restrict to subdomain
    region = mesh.Materials(domain_name)

    a = BilinearForm(fes)
    a += curl(u) * curl(v) * dx(definedon=region)

    m = BilinearForm(fes)
    m += u * v * dx(definedon=region)

    a.Assemble()
    m.Assemble()

    # Get freedofs
    freedofs = fes.FreeDofs()

    # Find DOFs with support in this domain
    domain_dofs = set()
    for el in mesh.Elements(VOL):
        if el.mat == domain_name:
            for d in fes.GetDofNrs(el):
                if d >= 0 and freedofs[d]:
                    domain_dofs.add(d)

    domain_dof_list = sorted(domain_dofs)
    n_domain_dofs = len(domain_dof_list)
    n_elements = len([el for el in mesh.Elements(VOL) if el.mat == domain_name])
    print(f"     Domain '{domain_name}': {n_domain_dofs} active DOFs, {n_elements} elements")

    if n_domain_dofs == 0:
        raise ValueError(f"No DOFs found in domain '{domain_name}'")

    # Extract submatrices for domain DOFs only
    A_dense = np.zeros((n_domain_dofs, n_domain_dofs), dtype=complex)
    M_dense = np.zeros((n_domain_dofs, n_domain_dofs), dtype=complex)

    tmp1 = GridFunction(fes)
    tmp2 = GridFunction(fes)

    for i, di in enumerate(domain_dof_list):
        tmp1.vec[:] = 0
        tmp1.vec[di] = 1

        tmp2.vec.data = a.mat * tmp1.vec
        for j, dj in enumerate(domain_dof_list):
            A_dense[j, i] = tmp2.vec[dj]

        tmp2.vec.data = m.mat * tmp1.vec
        for j, dj in enumerate(domain_dof_list):
            M_dense[j, i] = tmp2.vec[dj]

    # Regularize M if needed
    M_dense += 1e-14 * np.eye(n_domain_dofs)

    # Solve generalized eigenvalue problem
    from scipy.linalg import eigh
    eigenvalues, eigenvectors = eigh(A_dense, M_dense)

    # Filter out zero/negative eigenvalues
    valid_idx = eigenvalues > 1e-6
    eigenvalues = eigenvalues[valid_idx]
    eigenvectors = eigenvectors[:, valid_idx]

    if len(eigenvalues) == 0:
        raise ValueError(f"No valid eigenvalues found for domain '{domain_name}'")

    # Sort and take first n_modes
    sort_idx = np.argsort(eigenvalues)[:n_modes]
    eigenvalues = eigenvalues[sort_idx]
    eigenvectors = eigenvectors[:, sort_idx]

    # Convert to GridFunctions
    gf_modes = []
    for k in range(min(n_modes, eigenvectors.shape[1])):
        gf = GridFunction(fes, complex=True)
        gf.vec[:] = 0
        for i, di in enumerate(domain_dof_list):
            gf.vec[di] = eigenvectors[i, k]
        gf_modes.append(gf)

    return eigenvalues, gf_modes, domain_dof_list


# ============================================================================
# FIELD RECONSTRUCTION METHODS
# ============================================================================

def reconstruct_naive(fes, mesh, gf_list, domain_names):
    """
    NAIVE reconstruction using Set with definedon.
    This demonstrates the overwriting problem.
    """
    E_global = GridFunction(fes, complex=True)
    E_global.vec[:] = 0

    for gf, domain_name in zip(gf_list, domain_names):
        region = mesh.Materials(domain_name)
        E_global.Set(gf, definedon=region)

    return E_global


def reconstruct_direct_dof(fes, mesh, gf_list, domain_names, domain_dof_lists, scales=None):
    """
    CORRECT reconstruction using direct DOF assignment.
    """
    if scales is None:
        scales = [1.0] * len(gf_list)

    E_global = GridFunction(fes, complex=True)
    global_vec = E_global.vec.FV().NumPy()
    global_vec[:] = 0

    # Track contributions to each DOF
    dof_values = np.zeros(fes.ndof, dtype=complex)
    dof_counts = np.zeros(fes.ndof, dtype=int)

    for gf, domain_dofs, scale in zip(gf_list, domain_dof_lists, scales):
        local_vec = gf.vec.FV().NumPy()

        for dof in domain_dofs:
            dof_values[dof] += scale * local_vec[dof]
            dof_counts[dof] += 1

    # Assign values (average at interfaces)
    for dof in range(fes.ndof):
        if dof_counts[dof] > 0:
            global_vec[dof] = dof_values[dof] / dof_counts[dof]

    n_interface = np.sum(dof_counts > 1)
    print(f"     Direct DOF: {np.sum(dof_counts > 0)} DOFs set, {n_interface} interface DOFs")

    return E_global


def reconstruct_element_wise(fes, mesh, gf_list, domain_names, scales=None,
                             interface_mode='average'):
    """
    Element-wise reconstruction with explicit interface handling.
    """
    if scales is None:
        scales = [1.0] * len(gf_list)

    E_global = GridFunction(fes, complex=True)
    global_vec = E_global.vec.FV().NumPy()
    global_vec[:] = 0

    # Track contributions
    dof_contributions = {}

    for idx, (gf, domain_name, scale) in enumerate(zip(gf_list, domain_names, scales)):
        local_vec = gf.vec.FV().NumPy()

        for el in mesh.Elements(VOL):
            if el.mat != domain_name:
                continue

            for dof in fes.GetDofNrs(el):
                if dof < 0:
                    continue

                if dof not in dof_contributions:
                    dof_contributions[dof] = []

                # Check if this domain already contributed
                already = any(d_idx == idx for d_idx, _ in dof_contributions[dof])
                if not already:
                    dof_contributions[dof].append((idx, scale * local_vec[dof]))

    # Assign values
    n_interface = 0
    for dof, contributions in dof_contributions.items():
        if len(contributions) == 1:
            global_vec[dof] = contributions[0][1]
        else:
            n_interface += 1
            if interface_mode == 'average':
                global_vec[dof] = np.mean([c[1] for c in contributions])
            elif interface_mode == 'first':
                global_vec[dof] = contributions[0][1]
            elif interface_mode == 'last':
                global_vec[dof] = contributions[-1][1]

    print(f"     Element-wise ({interface_mode}): {len(dof_contributions)} DOFs, {n_interface} interface DOFs")

    return E_global


# ============================================================================
# INTERFACE SCALING
# ============================================================================

def compute_interface_scale(gf1, gf2, dofs1, dofs2, fes):
    """
    Compute scaling factor using shared interface DOFs.
    """
    interface_dofs = set(dofs1) & set(dofs2)
    print(f"     Found {len(interface_dofs)} interface DOFs")

    if not interface_dofs:
        return 1.0, 1.0

    vec1 = gf1.vec.FV().NumPy()
    vec2 = gf2.vec.FV().NumPy()

    vals1 = np.array([vec1[d] for d in interface_dofs])
    vals2 = np.array([vec2[d] for d in interface_dofs])

    # Filter near-zero
    mask = (np.abs(vals1) > 1e-12) | (np.abs(vals2) > 1e-12)
    if not np.any(mask):
        return 1.0, 1.0

    vals1 = vals1[mask]
    vals2 = vals2[mask]

    # Least squares: scale * vals2 ≈ vals1
    denom = np.vdot(vals2, vals2)
    if abs(denom) < 1e-14:
        return 1.0, 1.0

    scale = np.vdot(vals2, vals1) / denom

    mismatch_before = np.linalg.norm(vals1 - vals2) / (np.linalg.norm(vals1) + 1e-14)
    mismatch_after = np.linalg.norm(vals1 - scale * vals2) / (np.linalg.norm(vals1) + 1e-14)

    print(f"     Scale: {abs(scale):.4f} ∠{np.angle(scale)*180/np.pi:.1f}°")
    print(f"     Mismatch: {mismatch_before:.2%} → {mismatch_after:.2%}")

    return scale, mismatch_after


# ============================================================================
# DIAGNOSTICS
# ============================================================================

def compute_domain_norms(gf, mesh, domain_names):
    """Compute |E|² in each domain."""
    norms = {}
    for domain_name in domain_names:
        region = mesh.Materials(domain_name)
        norm_sq = Integrate(InnerProduct(gf, gf), mesh, definedon=region)
        norms[domain_name] = abs(norm_sq)
    return norms


def compare_interface_dofs(gf1, gf2, interface_dofs, n_show=10):
    """Compare field values at interface DOFs."""
    vec1 = gf1.vec.FV().NumPy()
    vec2 = gf2.vec.FV().NumPy()

    print(f"\n     Interface DOF comparison (first {n_show}):")
    print(f"     {'DOF':>6} | {'gf1':>24} | {'gf2':>24} | {'diff':>12}")
    print("     " + "-" * 75)

    for i, dof in enumerate(sorted(interface_dofs)[:n_show]):
        v1, v2 = vec1[dof], vec2[dof]
        diff = abs(v1 - v2)
        print(f"     {dof:>6} | {v1.real:>11.3e}{v1.imag:+11.3e}j | {v2.real:>11.3e}{v2.imag:+11.3e}j | {diff:>12.2e}")


# ============================================================================
# MAIN TEST
# ============================================================================

def run_test():
    """Main test function."""
    print("=" * 70)
    print("MULTI-DOMAIN FIELD RECONSTRUCTION TEST")
    print("=" * 70)

    # 1. Create geometry
    print("\n1. Creating split waveguide geometry...")
    mesh = create_split_waveguide_geometry(
        length1=1.0, length2=1.0,
        width=0.5, height=0.3,
        mesh_size=0.15
    )
    print(f"   Elements: {mesh.ne}, Vertices: {mesh.nv}")

    # Get materials
    el_to_mat, mat_names, mat_to_els = get_element_material_map(mesh)

    # 2. Create FE space
    print("\n2. Creating finite element space...")
    order = 2
    fes = HCurl(mesh, order=order, complex=True, dirichlet="wall")
    print(f"   FES: {fes.ndof} DOFs, order={order}")

    # 3. Solve eigenvalue problems
    print("\n3. Solving eigenvalue problems on each domain...")

    print("   Domain 1:")
    eigs1, modes1, dofs1 = solve_maxwell_eigenvalue_on_domain(fes, mesh, "domain1", n_modes=3)
    print(f"     Eigenvalues: {eigs1[:3]}")

    print("\n   Domain 2:")
    eigs2, modes2, dofs2 = solve_maxwell_eigenvalue_on_domain(fes, mesh, "domain2", n_modes=3)
    print(f"     Eigenvalues: {eigs2[:3]}")

    # Find interface DOFs
    interface_dofs = set(dofs1) & set(dofs2)
    print(f"\n   Interface DOFs: {len(interface_dofs)}")

    # 4. Test naive reconstruction
    print("\n4. Testing NAIVE reconstruction (Set with definedon)...")
    E_naive = reconstruct_naive(fes, mesh, [modes1[0], modes2[0]], ["domain1", "domain2"])
    norms_naive = compute_domain_norms(E_naive, mesh, ["domain1", "domain2"])
    print(f"   Norms: domain1={norms_naive['domain1']:.4e}, domain2={norms_naive['domain2']:.4e}")

    # 5. Test direct DOF reconstruction
    print("\n5. Testing DIRECT DOF reconstruction...")
    E_direct = reconstruct_direct_dof(
        fes, mesh,
        [modes1[0], modes2[0]],
        ["domain1", "domain2"],
        [dofs1, dofs2]
    )
    norms_direct = compute_domain_norms(E_direct, mesh, ["domain1", "domain2"])
    print(f"   Norms: domain1={norms_direct['domain1']:.4e}, domain2={norms_direct['domain2']:.4e}")

    # 6. Test element-wise reconstruction
    print("\n6. Testing ELEMENT-WISE reconstruction with different interface handling...")
    norms_elem = {}
    for mode in ['first', 'average', 'last']:
        E_elem = reconstruct_element_wise(
            fes, mesh,
            [modes1[0], modes2[0]],
            ["domain1", "domain2"],
            interface_mode=mode
        )
        norms = compute_domain_norms(E_elem, mesh, ["domain1", "domain2"])
        norms_elem[mode] = norms
        print(f"   {mode:>8}: domain1={norms['domain1']:.4e}, domain2={norms['domain2']:.4e}")

    # 7. Compute interface scaling
    print("\n7. Computing interface scaling...")
    scale, mismatch = compute_interface_scale(modes1[0], modes2[0], dofs1, dofs2, fes)

    # 8. Reconstruct with scaling
    print("\n8. Reconstruction WITH scaling...")
    E_scaled = reconstruct_direct_dof(
        fes, mesh,
        [modes1[0], modes2[0]],
        ["domain1", "domain2"],
        [dofs1, dofs2],
        scales=[1.0, scale]
    )
    norms_scaled = compute_domain_norms(E_scaled, mesh, ["domain1", "domain2"])
    print(f"   Norms: domain1={norms_scaled['domain1']:.4e}, domain2={norms_scaled['domain2']:.4e}")

    # 9. Interface analysis
    print("\n9. Interface DOF analysis...")
    compare_interface_dofs(modes1[0], modes2[0], interface_dofs)

    # 10. Visualize
    print("\n10. Visualization...")
    Draw(Norm(modes1[0]), mesh, "mode1_domain1")
    Draw(Norm(modes2[0]), mesh, "mode2_domain2")
    Draw(Norm(E_naive), mesh, "E_naive")
    Draw(Norm(E_direct), mesh, "E_direct")
    Draw(Norm(E_scaled), mesh, "E_scaled")

    # Summary
    print("\n" + "=" * 70)
    print("SUMMARY")
    print("=" * 70)
    print(f"{'Method':<25} | {'domain1':>15} | {'domain2':>15} | {'ratio':>10}")
    print("-" * 70)

    results_table = [
        ("Naive (Set/definedon)", norms_naive),
        ("Direct DOF (average)", norms_direct),
        ("Element-wise (first)", norms_elem['first']),
        ("Element-wise (average)", norms_elem['average']),
        ("Element-wise (last)", norms_elem['last']),
        ("Direct DOF + scaling", norms_scaled),
    ]

    for name, norms in results_table:
        n1, n2 = norms['domain1'], norms['domain2']
        ratio = n2 / n1 if n1 > 1e-15 else float('inf')
        print(f"{name:<25} | {n1:>15.4e} | {n2:>15.4e} | {ratio:>10.4f}")

    # Diagnostic
    print("\n" + "-" * 70)
    if norms_naive['domain1'] < 1e-10 or norms_naive['domain2'] < 1e-10:
        lost = 'domain1' if norms_naive['domain1'] < 1e-10 else 'domain2'
        print(f"⚠️  NAIVE method lost field in {lost}!")
        print("   This confirms the Set() overwriting problem.")
    else:
        print("ℹ️  Naive method preserved both domains (may work for non-overlapping DOFs)")

    if norms_direct['domain1'] > 1e-10 and norms_direct['domain2'] > 1e-10:
        print("✓  DIRECT DOF method preserves both domains!")

    print("=" * 70)

    return {
        'mesh': mesh,
        'fes': fes,
        'modes1': modes1,
        'modes2': modes2,
        'dofs1': dofs1,
        'dofs2': dofs2,
        'interface_dofs': interface_dofs,
        'E_naive': E_naive,
        'E_direct': E_direct,
        'E_scaled': E_scaled,
        'scale': scale,
        'norms': {
            'naive': norms_naive,
            'direct': norms_direct,
            'scaled': norms_scaled
        }
    }


if __name__ == "__main__":
    results = run_test()

MULTI-DOMAIN FIELD RECONSTRUCTION TEST

1. Creating split waveguide geometry...
   Elements: 364, Vertices: 158
   Materials in mesh: ['domain1', 'domain2']
   domain1: 182 elements
   domain2: 182 elements

2. Creating finite element space...
   FES: 4671 DOFs, order=2

3. Solving eigenvalue problems on each domain...
   Domain 1:
     Domain 'domain1': 1293 active DOFs, 182 elements
     Eigenvalues: [39.50809494 49.40493688 79.1874038 ]

   Domain 2:
     Domain 'domain2': 1293 active DOFs, 182 elements
     Eigenvalues: [39.50809494 49.40493688 79.1874038 ]

   Interface DOFs: 75

4. Testing NAIVE reconstruction (Set with definedon)...
   Norms: domain1=5.0612e-02, domain2=1.0000e+00

5. Testing DIRECT DOF reconstruction...
     Direct DOF: 2511 DOFs set, 75 interface DOFs
   Norms: domain1=1.0000e+00, domain2=9.9997e-01

6. Testing ELEMENT-WISE reconstruction with different interface handling...
     Element-wise (first): 4671 DOFs, 105 interface DOFs
      first: domain1=1.0000e+

WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.25…

WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.25…

WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.25…

WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.25…

WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.25…


SUMMARY
Method                    |         domain1 |         domain2 |      ratio
----------------------------------------------------------------------
Naive (Set/definedon)     |      5.0612e-02 |      1.0000e+00 |    19.7581
Direct DOF (average)      |      1.0000e+00 |      9.9997e-01 |     0.9999
Element-wise (first)      |      1.0000e+00 |      9.9995e-01 |     0.9999
Element-wise (average)    |      1.0000e+00 |      9.9997e-01 |     0.9999
Element-wise (last)       |      1.0001e+00 |      1.0000e+00 |     0.9999
Direct DOF + scaling      |      9.9996e-01 |      9.9740e-01 |     0.9974

----------------------------------------------------------------------
ℹ️  Naive method preserved both domains (may work for non-overlapping DOFs)
✓  DIRECT DOF method preserves both domains!
